## Gold Cascade Modeling (Deliverable 1.3.3)

This notebook implements the full three-stage cascade anomaly detection pipeline. The objective is to determine whether a layered approach improves alert quality when compared to the single-model baseline.

**Purpose:**  
To operationalize the cascade architecture defined in Section C, generating alert outputs for each stage of the cascade: broad detection (Stage 1), refined detection (Stage 2), and rule/profile/historical confirmation (Stage 3).

**Stages Implemented:**

1. **Stage 1 — Broad Isolation Forest**  
   High-sensitivity anomaly screen using all 50 Gold numeric features.

2. **Stage 2 — Narrow Isolation Forest**  
   Secondary detector trained on the reduced feature subset identified during Silver EDA.

3. **Stage 3 — Rule / Profile / Historical Confirmation**  
   Final confirmation based on behavior profiles, persistence checks, drift features, and cross-sensor consistency.

**Key Goals:**

- Load the Gold preprocessed dataset and Gold feature artifacts.
- Train Stage 1 and Stage 2 Isolation Forest models.
- Apply Stage 3 rule/profile confirmation logic based on Silver EDA outputs.
- Generate and store alert outputs for all three cascade stages.
- Export all cascade artifacts for comparative evaluation.

**Relevance to Section C:**  
This notebook produces the layered alert outputs required for evaluating the cascade’s effect on false positives, noisy alerts, and anomaly sensitivity. These outputs are necessary for the statistical tests, alert-volume comparisons, and visual communication described in Section C.

## Overview

This notebook builds the first staged cascade anomaly-detection workflow for comparison against the baseline. It is part of the Gold portion of the capstone workflow and should be read as a notebook-first implementation step rather than a separate pipeline redesign.


## Objectives

- Document how this notebook builds the first staged cascade anomaly-detection workflow for comparison against the baseline.
- Preserve the existing notebook-first execution flow, configuration usage, logger behavior, ledger behavior, and artifact outputs.
- Make the notebook easier to review by separating setup, validation, processing, outputs, and handoff context.
- Connect the modeling or validation outputs back to the final anomaly-detection evidence used in the capstone write-up.


## Code Reference

A detailed code-cell reference for this notebook is maintained in the project documentation:`docs/technical_reference/notebook_code_references/EDA_Notebook_Pump_Gold_03a_Cascade_Modeling_code_reference.md`


## Cascade Modeling Setup and Imports

In this section I am loading the libraries and project utilities needed for the Gold cascade modeling stage.

The purpose here is to get the notebook ready before I start working with the Gold modeling inputs. That includes:
- standard Python libraries
- model and metrics utilities
- path and config loading
- logging
- truth-record helpers
- artifact saving utilities
- experiment tracking support

At this point I am not fitting the cascade yet. I am just preparing the notebook so the full multi-stage workflow can run in a structured and repeatable way.

In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union, Mapping, cast

from pathlib import Path
import yaml
import os

import logging
import wandb

import pandas as pd 
import numpy as np 

import joblib 

from sklearn.model_selection import (
    train_test_split, 
    KFold,
)

from sklearn.preprocessing import (
    StandardScaler, 
    MinMaxScaler, 
    OneHotEncoder, 
    RobustScaler,
)

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import (
    RandomForestClassifier, 
    IsolationForest,
)

from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    precision_recall_fscore_support, 
    roc_auc_score, 
    average_precision_score,
)

from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor

# Custom Utilities Module
from utils.core.paths import get_paths
from utils.core.file_io import (
    load_data, 
    save_data, 
    save_json, 
    load_json,
)

from utils.core.logging_profiler import profile_dataframe

from utils.core.logging_setup import (
    configure_logging, 
    log_layer_paths,
)

from utils.core.wandb_utils import finalize_wandb_stage

from utils.core.truths import (
    make_process_run_id,
    build_file_fingerprint,
    extract_truth_hash,
    identify_meta_columns,
    identify_feature_columns,
    initialize_layer_truth,
    update_truth_section,
    build_truth_record,
    save_truth_record,
    append_truth_index,
    stamp_truth_columns,
    load_truth_record,
    find_truth_record_by_hash,
    load_truth_record_by_hash,
    load_parent_truth_record_from_dataframe,
    get_truth_value,
    get_dataset_name_from_truth,
    get_truth_hash,
    get_parent_truth_hash,
    get_pipeline_mode_from_truth,
    get_artifact_path_from_truth,
)
from utils.core.config_loader import (
    build_truth_config_block,
    set_wandb_dir_from_config,
    export_config_snapshot,
    load_pipeline_config,
)

from utils.medallion.gold.cascade_row_tracking import (
    ensure_stable_row_id,
    build_stage_scoring_frame,
    score_isolation_forest_stage,
    merge_stage_results_back,
    finalize_stage_flag_columns,
    get_detected_rows_dataframe,
    get_stage_detected_rows_dataframe,
)

from utils.medallion.gold.gold_cascade_validation_contracts import (
    build_cascade_variant_contract,
    build_stage3_rule_payload_from_globals,
    write_json_contract,
)

from utils.core.artifacts import gold_model_validation_contract_path
from utils.medallion.gold.gold_cascade_validation_contracts import (
    build_gold_model_output_validation_contract,
    write_gold_model_output_validation_contract,
)

from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
)
from utils.database.layer_postgres import (
    read_layer_dataframe, 
    write_layer_dataframe, 
    prepare_layer_dataframe,
)

from utils.database.sql_notebook_helpers import (
    delete_dataset_run_rows,
    execute_many,
    get_existing_dataframe,
    get_row_value,
    log_data_quality_event,
    log_pipeline_artifact,
    preview_sql_table,
    row_to_payload,
    sql_table_ref,
    to_builtin,
    to_json_string,
    to_scalar,
    upsert_pipeline_run,
)

from utils.database.medallion_sql_writers import (
    log_gold_05_anomaly_detection_summary_sql,
    log_silver_eda_sql,
    write_bronze_sensor_observations_sql,
    write_gold_baseline_scores_sql,
    write_gold_cascade_scores_sql,
    write_gold_model_comparison_results_sql,
    write_gold_preprocessed_features_sql,
    write_silver_sensor_observation_features_sql,
)

# Ledger 
from utils.core.ledger import Ledger

from utils.core.artifacts import (
    build_artifact_dirs_from_config,
    artifact_file_path,
)

from utils.core.notebook_context import load_notebook_context

# Show more columns
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)


----

In [2]:
def choose_threshold_by_percentile(
    scores: Sequence[float],
    percentile: float = 95.0,
    *,
    return_info: bool = False,
) -> float | Tuple[float, Dict[str, Any]]:
    """
    Choose anomaly threshold using a score percentile.

    Compatibility behavior
    ----------------------
    - Notebook-style usage:
    
        threshold = choose_threshold_by_percentile(scores, 95.0)
      returns a float threshold.
    - Pipeline-style usage:
        threshold, info = choose_threshold_by_percentile(scores, 95.0, return_info=True)
      returns both threshold and metadata.
    """
    scores_array = np.asarray(scores, dtype=float)

    if scores_array.size == 0:
        raise ValueError("Cannot choose threshold from empty score array.")

    threshold = float(np.percentile(scores_array, float(percentile)))

    info = {
        "threshold_method": "percentile",
        "percentile": float(percentile),
        "threshold": threshold,
        "score_count": int(scores_array.size),
        "score_min": float(np.min(scores_array)),
        "score_max": float(np.max(scores_array)),
        "score_mean": float(np.mean(scores_array)),
    }

    if return_info:
        return threshold, info
    return threshold

#### Define configuration mapping guards

This cell defines helper logic used by the surrounding `----` section. Keeping the helper directly before its use makes the notebook assumptions visible and easier to audit.

In [3]:
def cfg_require_mapping(value: object, name: str) -> Mapping[str, Any]:
    if not isinstance(value, Mapping):
        raise TypeError(
            f"{name} must be a mapping, got {type(value).__name__}: {value!r}"
        )

    return cast(Mapping[str, Any], value)


def cfg_optional_mapping(value: object | None, name: str) -> Mapping[str, Any]:
    if value is None:
        return {}

    return cfg_require_mapping(value, name)


def require_mapping(value: Any, name: str) -> dict[str, Any]:
    """
    Validate that a loaded JSON/config object is a dictionary.
    """
    if not isinstance(value, dict):
        raise TypeError(
            f"{name} must be a dictionary. "
            f"Got {type(value).__name__}: {value!r}"
        )

    return value


def require_str_list(value: Any, name: str) -> list[str]:
    """
    Validate that a loaded JSON/config object is a list of strings.
    """
    if value is None:
        raise ValueError(f"{name} is None.")

    if not isinstance(value, (list, tuple)):
        raise TypeError(
            f"{name} must be a list/tuple of column names. "
            f"Got {type(value).__name__}: {value!r}"
        )

    cleaned_values = [
        str(item).strip()
        for item in value
        if str(item).strip()
    ]

    if not cleaned_values:
        raise ValueError(f"{name} resolved to an empty list.")

    return cleaned_values


def require_float(value: Any, name: str) -> float:
    """
    Convert a scalar or threshold-return tuple into a float.

    Some project helpers may return either:
        threshold
    or:
        (threshold, metadata)
    """
    if isinstance(value, tuple):
        if len(value) == 0:
            raise ValueError(f"{name} tuple is empty.")
        value = value[0]

    return float(value)


def as_bool_array(value: Any, name: str) -> np.ndarray:
    """
    Convert a Pandas/NumPy boolean mask into a NumPy bool array.
    """
    if isinstance(value, pd.Series):
        return value.to_numpy(dtype=bool)

    return np.asarray(value, dtype=bool)


def as_int_array(value: Any, name: str) -> np.ndarray:
    """
    Convert labels/flags into a NumPy int array.
    """
    if value is None:
        raise ValueError(f"{name} is None.")

    if isinstance(value, pd.Series):
        return value.fillna(0).astype(int).to_numpy(dtype=int)

    return np.asarray(value, dtype=int)


def as_float_array(value: Any, name: str) -> np.ndarray:
    """
    Convert scores into a flat NumPy float array.
    """
    if value is None:
        raise ValueError(f"{name} is None.")

    return np.asarray(value, dtype=float).reshape(-1)


def choose_threshold_value(scores: Any, percentile: float) -> float:
    """
    Normalize score input and threshold helper output for Pylance.
    """
    score_values = as_float_array(scores, "scores")
    threshold_result = choose_threshold_by_percentile(
        score_values.tolist(),
        percentile,
    )
    return require_float(threshold_result, "threshold")

---

## Load Configuration, Paths, and Cascade Runtime Settings

### Ask

What cascade settings am I resolving here?

### Answer

This cell loads the configuration values that control the default cascade run.

That includes dataset identity, Gold artifact paths, Stage 1 settings, Stage 2 fixed-threshold settings, Stage 3 rule thresholds, truth locations, run metadata, and output file paths. I want this resolved near the top so the cascade logic is driven by configuration instead of scattered hard-coded values.

In [4]:


# Shared notebook context
CONTEXT_STAGE = "gold_cascade"
CONTEXT_DATASET = "pump"
CONTEXT_LAYER = "gold"
CONFIG_RUN_MODE = "train"
CONFIG_PROFILE = "default"
CONTEXT_LOG_FILE = "gold_modeling_cascade_defaults.log"

CTX = load_notebook_context(
    stage=CONTEXT_STAGE,
    dataset=CONTEXT_DATASET,
    mode=CONFIG_RUN_MODE,
    profile=CONFIG_PROFILE,
    logger_child_name="capstone.gold.cascade.defaults",
    log_filename=CONTEXT_LOG_FILE,
)

# Shared aliases used throughout the notebook
paths = CTX.paths
CONFIG = CTX.config
CONFIG_MAP = CTX.config
STAGE_CFG = CTX.stage_config
GOLD_CFG = CTX.stage_config
RESOLVED_PATHS = CTX.resolved_paths
FILENAMES = CTX.filenames
VERSIONS_CFG = CTX.versions
RUNTIME_CFG = CTX.runtime
DATASET_CFG = CTX.dataset_config
WANDB_CFG = CTX.wandb
EXECUTION_CFG = CTX.execution
PIPELINE = CTX.pipeline
logger = CTX.logger
ledger = CTX.ledger
LOG_PATH = CTX.log_path
CONTEXT_RECIPE_ID = CTX.recipe_id

logger.info(
    "Notebook context loaded",
    extra={
        "stage": CONTEXT_STAGE,
        "recipe_id": CONTEXT_RECIPE_ID,
        "dataset": CONTEXT_DATASET,
        "mode": CONFIG_RUN_MODE,
        "profile": CONFIG_PROFILE,
    },
)

ledger.add(
    kind="step",
    step="context_loaded",
    message="Loaded shared notebook context.",
    data={
        "stage": CONTEXT_STAGE,
        "recipe_id": CONTEXT_RECIPE_ID,
        "dataset": CONTEXT_DATASET,
        "mode": CONFIG_RUN_MODE,
        "profile": CONFIG_PROFILE,
        "log_path": str(LOG_PATH),
    },
    logger=logger,
)


2026-06-16 18:10:18,722 | INFO | capstone.gold.cascade.defaults | gold_cascade stage starting
2026-06-16 18:10:18,725 | INFO | capstone.gold.cascade.defaults | LEDGER | {'ts_utc': '2026-06-16T18:10:18.725395+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'init', 'message': 'Initialized ledger from shared notebook context', 'why': None, 'consequence': None, 'data': {'stage': 'gold_cascade', 'recipe_id': 'gold_modeling__v001_cascade', 'dataset': 'pump', 'mode': 'train', 'profile': 'default', 'log_path': '/workspace/logs/gold_modeling_cascade_defaults.log'}}
2026-06-16 18:10:18,728 | INFO | capstone.gold.cascade.defaults | Notebook context loaded
2026-06-16 18:10:18,730 | INFO | capstone.gold.cascade.defaults | LEDGER | {'ts_utc': '2026-06-16T18:10:18.730465+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'context_loaded', 'message': 'Loaded shared notebook context.', 'why': None, 'consequ

{'ts_utc': '2026-06-16T18:10:18.730465+00:00',
 'stage': 'gold_cascade',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'context_loaded',
 'message': 'Loaded shared notebook context.',
 'why': None,
 'consequence': None,
 'data': {'stage': 'gold_cascade',
  'recipe_id': 'gold_modeling__v001_cascade',
  'dataset': 'pump',
  'mode': 'train',
  'profile': 'default',
  'log_path': '/workspace/logs/gold_modeling_cascade_defaults.log'}}

In [5]:
# Fixed/default cascade notebook selector.
CASCADE_VARIANT = "default"

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####

TRUTH_CONFIG = cast(Dict[str, Any], build_truth_config_block(CONFIG))
TRUTH_CONFIG["pipeline"] = PIPELINE

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- Stage details ----

STAGE = "gold"

LAYER_NAME = str(GOLD_CFG["layer_name"])

RECIPE_ID = str(GOLD_CFG["recipe_id"])

GOLD_VERSION = str(VERSIONS_CFG["gold"])
TRUTH_VERSION = str(VERSIONS_CFG["truth"])

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####

PIPELINE_MODE = str(PIPELINE["execution_mode"])
RUN_MODE = str(RUNTIME_CFG.get("mode", CONFIG_RUN_MODE))
CONFIG_PROFILE = str(RUNTIME_CFG.get("profile", CONFIG_PROFILE))

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####

DATASET_NAME_CONFIG = str(DATASET_CFG.get("name", "pump"))
DATASET_NAME = DATASET_NAME_CONFIG.strip().lower()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####

GOLD_PROCESS_RUN_ID = make_process_run_id(
    str(GOLD_CFG["process_run_id_prefix"])
)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- W&B ----

WANDB_PROJECT = str(WANDB_CFG.get("project", "capstone"))
WANDB_ENTITY = str(WANDB_CFG.get("entity", ""))
WANDB_RUN_NAME = f"{GOLD_VERSION}"

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- File names ----

GOLD_PREPROCESSED_FILE_NAME = str(FILENAMES["gold_preprocessed_file_name"])
GOLD_PREPROCESSED_SCALED_FILE_NAME = str(
    FILENAMES["gold_preprocessed_scaled_file_name"]
)

GOLD_FIT_FILE_NAME = str(FILENAMES["gold_fit_file_name"])
GOLD_TRAIN_FILE_NAME = str(FILENAMES["gold_train_file_name"])
GOLD_TEST_FILE_NAME = str(FILENAMES["gold_test_file_name"])

STAGE1_FEATURES_FILE_NAME = str(FILENAMES["stage1_features_file_name"])
STAGE2_FEATURES_FILE_NAME = str(FILENAMES["stage2_features_file_name"])
STAGE3_PRIMARY_FILE_NAME = str(FILENAMES["stage3_primary_file_name"])
STAGE3_SECONDARY_FILE_NAME = str(FILENAMES["stage3_secondary_file_name"])

STAGE1_MODEL_FILE_NAME = str(
    FILENAMES["cascade_defaults_stage1_model_file_name"]
)
STAGE2_MODEL_FILE_NAME = str(
    FILENAMES["cascade_defaults_stage2_model_file_name"]
)

CASCADE_THRESHOLDS_FILE_NAME = str(
    FILENAMES["cascade_defaults_thresholds_file_name"]
)
CASCADE_SUMMARY_FILE_NAME = str(
    FILENAMES["cascade_defaults_summary_file_name"]
)
CASCADE_METADATA_FILE_NAME = str(
    FILENAMES["cascade_defaults_metadata_file_name"]
)

CASCADE_RESULTS_FILE_NAME_CSV = str(
    FILENAMES["cascade_defaults_results_file_name_csv"]
)
CASCADE_RESULTS_FILE_NAME_PICKLE = str(
    FILENAMES["cascade_defaults_results_file_name_pickle"]
)

CASCADE_REFERENCE_PROFILE_FILE_NAME = str(
    FILENAMES["cascade_defaults_reference_profile_file_name"]
)

GOLD_CASCADE_LEDGER_FILE_NAME = str(
    FILENAMES.get(
        "gold_cascade_defaults_ledger_file_name",
        "gold_cascade_defaults_ledger.jsonl",
    )
)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- Cascade config blocks ----

STAGE1_CFG = cast(
    Dict[str, Any],
    cfg_require_mapping(GOLD_CFG.get("stage1", {}), "CONFIG['gold_cascade']['stage1']"),
)

STAGE2_CFG = cast(
    Dict[str, Any],
    cfg_require_mapping(GOLD_CFG.get("stage2", {}), "CONFIG['gold_cascade']['stage2']"),
)

STAGE3_CFG = cast(
    Dict[str, Any],
    cfg_require_mapping(GOLD_CFG.get("stage3", {}), "CONFIG['gold_cascade']['stage3']"),
)

STAGE2_FIXED_CFG = cast(
    Dict[str, Any],
    cfg_require_mapping(
        STAGE2_CFG.get("fixed", {}),
        "CONFIG['gold_cascade']['stage2']['fixed']",
    ),
)

STAGE2_FIXED_PARAMS = cast(
    Dict[str, Any],
    cfg_require_mapping(
        STAGE2_FIXED_CFG.get("params", {}),
        "CONFIG['gold_cascade']['stage2']['fixed']['params']",
    ),
)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- Runtime knobs ----

RANDOM_SEED = int(GOLD_CFG["random_seed"])

STAGE1_ESTIMATOR_COUNT = int(STAGE1_CFG["estimator_count"])
STAGE1_THRESHOLD_PERCENTILE = float(STAGE1_CFG["threshold_percentile"])

# Fixed-only notebook.
STAGE2_SELECTION_MODE = "fixed"
STAGE2_RANDOM_STATE = int(STAGE2_CFG.get("random_state", RANDOM_SEED))

STAGE2_FIXED_THRESHOLD_PERCENTILE = float(
    STAGE2_FIXED_CFG["threshold_percentile"]
)

STAGE2_FIXED_WARNING_THRESHOLD_PERCENTILE = float(
    STAGE2_FIXED_CFG.get(
        "warning_threshold_percentile",
        STAGE2_FIXED_CFG["threshold_percentile"],
    )
)

STAGE2_FIXED_CONFIRMED_THRESHOLD_PERCENTILE = float(
    STAGE2_FIXED_CFG.get(
        "confirmed_threshold_percentile",
        STAGE2_FIXED_CFG["threshold_percentile"],
    )
)

STAGE3_MIN_PRIMARY_SENSOR_HITS = int(STAGE3_CFG["min_primary_sensor_hits"])
STAGE3_MIN_SECONDARY_SENSOR_HITS = int(STAGE3_CFG["min_secondary_sensor_hits"])
STAGE3_ROLLING_WINDOW_SIZE = int(STAGE3_CFG["rolling_window_size"])
STAGE3_MINIMUM_FLAGS_IN_WINDOW = int(STAGE3_CFG["minimum_flags_in_window"])

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- Base paths only ----

ARTIFACTS_ROOT = Path(str(RESOLVED_PATHS["artifacts_root"]))

GOLD_PREPROCESSED_DATA_PATH = Path(
    str(RESOLVED_PATHS["gold_preprocessed_data_path"])
)
GOLD_PREPROCESSED_SCALED_DATA_PATH = Path(
    str(RESOLVED_PATHS["gold_preprocessed_scaled_data_path"])
)

GOLD_TRAIN_DATA_PATH = Path(str(RESOLVED_PATHS["gold_train_data_path"]))
GOLD_TEST_DATA_PATH = Path(str(RESOLVED_PATHS["gold_test_data_path"]))
GOLD_FIT_DATA_PATH = Path(str(RESOLVED_PATHS["gold_fit_data_path"]))

GOLD_ARTIFACTS_PATH = Path(str(RESOLVED_PATHS["gold_artifacts_dir"]))

STAGE1_FEATURES_PATH = Path(str(RESOLVED_PATHS["stage1_features_path"]))
STAGE2_FEATURES_PATH = Path(str(RESOLVED_PATHS["stage2_features_path"]))
STAGE3_PRIMARY_PATH = Path(str(RESOLVED_PATHS["stage3_primary_path"]))
STAGE3_SECONDARY_PATH = Path(str(RESOLVED_PATHS["stage3_secondary_path"]))

MODELS_PATH = Path(str(RESOLVED_PATHS["models_root"]))

STAGE1_MODELS_PATH = Path(
    str(RESOLVED_PATHS["cascade_defaults_stage1_models_path"])
)
STAGE1_MODEL_ARTIFACT_PATH = Path(
    str(RESOLVED_PATHS["cascade_defaults_stage1_model_artifact_path"])
)

STAGE2_MODELS_PATH = Path(
    str(RESOLVED_PATHS["cascade_defaults_stage2_models_path"])
)
STAGE2_MODEL_ARTIFACT_PATH = Path(
    str(RESOLVED_PATHS["cascade_defaults_stage2_model_artifact_path"])
)

CASCADE_RESULTS_PATH_CSV = Path(
    str(RESOLVED_PATHS["cascade_defaults_results_path_csv"])
)
CASCADE_RESULTS_PATH_PICKLE = Path(
    str(RESOLVED_PATHS["cascade_defaults_results_path_pickle"])
)

CASCADE_THRESHOLDS_PATH = Path(
    str(RESOLVED_PATHS["cascade_defaults_thresholds_path"])
)
CASCADE_SUMMARY_PATH = Path(
    str(RESOLVED_PATHS["cascade_defaults_summary_path"])
)
CASCADE_METADATA_PATH = Path(
    str(RESOLVED_PATHS["cascade_defaults_metadata_path"])
)

CASCADE_REFERENCE_PROFILE_PATH = Path(
    str(RESOLVED_PATHS["cascade_defaults_reference_profile_path"])
)

TRUTHS_PATH = Path(str(RESOLVED_PATHS["truths_dir"]))
TRUTH_INDEX_PATH = Path(str(RESOLVED_PATHS["truth_index_path"]))

LOGS_PATH = Path(str(RESOLVED_PATHS["logs_root"]))

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# W&B

set_wandb_dir_from_config(CONFIG)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# Path failsafes

GOLD_PREPROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_PREPROCESSED_SCALED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

GOLD_FIT_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_TEST_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_TRAIN_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

GOLD_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)

STAGE1_FEATURES_PATH.parent.mkdir(parents=True, exist_ok=True)
STAGE2_FEATURES_PATH.parent.mkdir(parents=True, exist_ok=True)
STAGE3_PRIMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
STAGE3_SECONDARY_PATH.parent.mkdir(parents=True, exist_ok=True)

MODELS_PATH.mkdir(parents=True, exist_ok=True)

STAGE1_MODELS_PATH.parent.mkdir(parents=True, exist_ok=True)
#STAGE1_MODEL_ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)

STAGE2_MODELS_PATH.parent.mkdir(parents=True, exist_ok=True)
#STAGE2_MODEL_ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)

CASCADE_RESULTS_PATH_CSV.parent.mkdir(parents=True, exist_ok=True)
CASCADE_RESULTS_PATH_PICKLE.parent.mkdir(parents=True, exist_ok=True)

CASCADE_THRESHOLDS_PATH.parent.mkdir(parents=True, exist_ok=True)
CASCADE_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
CASCADE_METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)

CASCADE_REFERENCE_PROFILE_PATH.parent.mkdir(parents=True, exist_ok=True)

TRUTHS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####

print("Project root:", paths.root)
print("Artifacts root:", ARTIFACTS_ROOT)
print("CONFIG dataset:", DATASET_NAME_CONFIG)
print("Cascade variant:", CASCADE_VARIANT)
print("Gold fit path:", GOLD_FIT_DATA_PATH)
print("Stage 1 model path:", STAGE1_MODEL_ARTIFACT_PATH)
print("Stage 2 model path:", STAGE2_MODEL_ARTIFACT_PATH)
print("Cascade results path:", CASCADE_RESULTS_PATH_CSV)

Project root: /workspace
Artifacts root: /workspace/artifacts
CONFIG dataset: pump
Cascade variant: default
Gold fit path: /workspace/data/gold/pump__gold__fit_normal_only.parquet
Stage 1 model path: /workspace/artifacts/gold/pump/cascade_defaults/models/pump__gold__cascade_defaults_stage1_isolation_forest.joblib
Stage 2 model path: /workspace/artifacts/gold/pump/cascade_defaults/models/pump__gold__cascade_defaults_stage2_isolation_forest.joblib
Cascade results path: /workspace/artifacts/gold/pump/cascade_defaults/scores/pump__gold__cascade_defaults_results.csv


----

In [6]:
required_context_vars = [
    "CTX",
    "paths",
    "CONFIG",
    "CONFIG_MAP",
    "STAGE_CFG",
    "RESOLVED_PATHS",
    "FILENAMES",
    "VERSIONS_CFG",
    "RUNTIME_CFG",
    "DATASET_CFG",
    "WANDB_CFG",
    "EXECUTION_CFG",
    "PIPELINE",
    "logger",
    "ledger",
    "LOG_PATH",
]

missing_context_vars = [name for name in required_context_vars if name not in globals()]
if missing_context_vars:
    raise NameError(f"Missing required shared context variables: {missing_context_vars}")

logger.info(
    "Context sanity check passed",
    extra={
        "stage": CTX.stage,
        "recipe_id": CTX.recipe_id,
        "dataset": CTX.dataset,
        "mode": CTX.mode,
        "profile": CTX.profile,
        "log_path": str(CTX.log_path),
    },
)

ledger.add(
    kind="check",
    step="context_sanity_check",
    message="Verified shared notebook context variables are available.",
    data={
        "stage": CTX.stage,
        "recipe_id": CTX.recipe_id,
        "dataset": CTX.dataset,
        "mode": CTX.mode,
        "profile": CTX.profile,
        "log_path": str(CTX.log_path),
    },
    logger=logger,
)

pd.DataFrame(
    [
        {"name": name, "status": "present"}
        for name in required_context_vars
    ]
)

2026-06-16 18:10:19,273 | INFO | capstone.gold.cascade.defaults | Context sanity check passed
2026-06-16 18:10:19,277 | INFO | capstone.gold.cascade.defaults | LEDGER | {'ts_utc': '2026-06-16T18:10:19.277331+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'check', 'step': 'context_sanity_check', 'message': 'Verified shared notebook context variables are available.', 'why': None, 'consequence': None, 'data': {'stage': 'gold_cascade', 'recipe_id': 'gold_modeling__v001_cascade', 'dataset': 'pump', 'mode': 'train', 'profile': 'default', 'log_path': '/workspace/logs/gold_modeling_cascade_defaults.log'}}


,name,status
0,CTX,present
1,paths,present
2,CONFIG,present
3,CONFIG_MAP,present
4,STAGE_CFG,present
5,RESOLVED_PATHS,present
6,FILENAMES,present
7,VERSIONS_CFG,present
8,RUNTIME_CFG,present
9,DATASET_CFG,present


In [7]:
gold_required_context_vars = [
    "GOLD_CFG",
]

missing_gold_context_vars = [
    name for name in gold_required_context_vars
    if name not in globals()
]

if missing_gold_context_vars:
    raise NameError(f"Missing Gold context variables: {missing_gold_context_vars}")

logger.info("Gold context sanity check passed")

2026-06-16 18:10:19,333 | INFO | capstone.gold.cascade.defaults | Gold context sanity check passed


---

In [8]:
CASCADE_VARIANT = "default"

GOLD_CASCADE_ARTIFACT_DIRS = build_artifact_dirs_from_config(
    config=CONFIG,
    stage_key="gold_cascade",
    variant=CASCADE_VARIANT,
)

GOLD_ARTIFACTS_PATH = GOLD_CASCADE_ARTIFACT_DIRS["stage_dataset_root"]
GOLD_CASCADE_ROOT = GOLD_CASCADE_ARTIFACT_DIRS["root"]

STAGE1_MODEL_ARTIFACT_PATH = (
    GOLD_CASCADE_ARTIFACT_DIRS["models"]
    / FILENAMES["cascade_defaults_stage1_model_file_name"]
)

STAGE2_MODEL_ARTIFACT_PATH = (
    GOLD_CASCADE_ARTIFACT_DIRS["models"]
    / FILENAMES["cascade_defaults_stage2_model_file_name"]
)

#STAGE1_MODELS_PATH = STAGE1_MODEL_ARTIFACT_PATH
#STAGE2_MODELS_PATH = STAGE2_MODEL_ARTIFACT_PATH

CASCADE_RESULTS_PATH_CSV = (
    GOLD_CASCADE_ARTIFACT_DIRS["scores"]
    / FILENAMES["cascade_defaults_results_file_name_csv"]
)

CASCADE_RESULTS_PATH_PICKLE = (
    GOLD_CASCADE_ARTIFACT_DIRS["scores"]
    / FILENAMES["cascade_defaults_results_file_name_pickle"]
)

CASCADE_THRESHOLDS_PATH = (
    GOLD_CASCADE_ARTIFACT_DIRS["thresholds"]
    / FILENAMES["cascade_defaults_thresholds_file_name"]
)

CASCADE_SUMMARY_PATH = (
    GOLD_CASCADE_ARTIFACT_DIRS["summaries"]
    / FILENAMES["cascade_defaults_summary_file_name"]
)

CASCADE_METADATA_PATH = (
    GOLD_CASCADE_ARTIFACT_DIRS["metadata"]
    / FILENAMES["cascade_defaults_metadata_file_name"]
)

CASCADE_REFERENCE_PROFILE_PATH = (
    GOLD_CASCADE_ARTIFACT_DIRS["profiles"]
    / FILENAMES["cascade_defaults_reference_profile_file_name"]
)

cascade_ledger_path = (
    GOLD_CASCADE_ARTIFACT_DIRS["lineage"]
    / FILENAMES["gold_cascade_defaults_ledger_file_name"]
)

CASCADE_ROW_TRACKING_DIR = GOLD_CASCADE_ARTIFACT_DIRS["row_tracking"]
CASCADE_PLOT_DIR = GOLD_CASCADE_ARTIFACT_DIRS["plots"]
GOLD_CASCADE_CONFIG_DIR = GOLD_CASCADE_ARTIFACT_DIRS["config"]

CONFIG_SNAPSHOT_PATH = (
    GOLD_CASCADE_CONFIG_DIR
    / f"{DATASET_NAME}__gold_cascade_defaults__resolved_config.yaml"
)

if CONFIG["execution"].get("save_config_snapshot", True):
    export_config_snapshot(CONFIG, CONFIG_SNAPSHOT_PATH)

----

#### Review intermediate output

This cell displays an intermediate checkpoint. Use the output to confirm that the expected rows, columns, dtypes, or summary values are present before continuing.

In [9]:
# =============================================================================
# SQL Runtime Context
# Purpose:
#   Establish the Postgres connection and resolve the dataset/run identifiers
#   used by SQL logging and medallion table writes.
# =============================================================================

engine = get_engine_from_env()

CAPSTONE_SCHEMA: str = str(os.getenv("CAPSTONE_SCHEMA", "capstone"))


def first_non_empty_string(*values: object) -> str | None:
    """
    Return the first non-empty string-like value from a list of candidates.

    This helper skips None, empty strings, whitespace-only strings, and
    dictionaries. It is used to resolve dataset/run identifiers without
    accidentally accepting an invalid config object or a placeholder None value.
    """
    for value in values:
        if value is None:
            continue

        if isinstance(value, dict):
            continue

        text_value = str(value).strip()

        if text_value:
            return text_value

    return None


dataset_config = (
    cast(Dict[str, Any], CONFIG.get("dataset", {}))
    if isinstance(CONFIG.get("dataset", {}), dict)
    else {}
)

dataset_config_name = dataset_config.get("name")
dataset_config_id = dataset_config.get("dataset_id", dataset_config.get("id"))
dataset_config_run_id = dataset_config.get("run_id")
dataset_config_asset_id = dataset_config.get("asset_id")

is_synthetic_run = str(RUN_MODE).lower() in {
    "synthetic",
    "synthetic_train",
    "synthetic_run",
    "simulation",
}

if is_synthetic_run:
    raw_dataset_id = first_non_empty_string(
        os.getenv("CAPSTONE_DATASET_ID"),
        os.getenv("SYNTHETIC_DATASET_ID"),
        globals().get("DATASET_ID"),
        dataset_config_id,
        globals().get("DATASET_NAME_CONFIG"),
        dataset_config_name,
        "pump_synthetic_v1",
    )

    raw_run_id = first_non_empty_string(
        os.getenv("CAPSTONE_RUN_ID"),
        os.getenv("SYNTHETIC_RUN_ID"),
        globals().get("DATASET_RUN_ID"),
        dataset_config_run_id,
        globals().get("RUN_ID"),
        globals().get("RUN_ID_DEFAULT_FALLBACK"),
        "synthetic_run_001",
    )

    raw_asset_id = first_non_empty_string(
        os.getenv("CAPSTONE_ASSET_ID"),
        os.getenv("SYNTHETIC_ASSET_ID"),
        globals().get("ASSET_ID"),
        dataset_config_asset_id,
        globals().get("ASSET_ID_DEFAULT_FALLBACK"),
        "pump_asset_001",
    )

else:
    raw_dataset_id = first_non_empty_string(
        os.getenv("CAPSTONE_DATASET_ID"),
        globals().get("DATASET_ID"),
        dataset_config_id,
        globals().get("DATASET_NAME_CONFIG"),
        dataset_config_name,
        "pump",
    )

    raw_run_id = first_non_empty_string(
        os.getenv("CAPSTONE_RUN_ID"),
        globals().get("DATASET_RUN_ID"),
        dataset_config_run_id,
        globals().get("RUN_ID"),
        globals().get("RUN_ID_DEFAULT_FALLBACK"),
        "run__001",
    )

    raw_asset_id = first_non_empty_string(
        os.getenv("CAPSTONE_ASSET_ID"),
        globals().get("ASSET_ID"),
        dataset_config_asset_id,
        globals().get("ASSET_ID_DEFAULT_FALLBACK"),
        "asset__001",
    )


if raw_dataset_id is None:
    raise ValueError(
        "DATASET_ID could not be resolved. "
        "Set CAPSTONE_DATASET_ID or configure CONFIG['dataset']['name'] / "
        "CONFIG['dataset']['dataset_id']."
    )

if raw_run_id is None:
    raise ValueError(
        "RUN_ID could not be resolved. "
        "Set CAPSTONE_RUN_ID, CONFIG['dataset']['run_id'], or default_fallbacks.run_id."
    )

if raw_asset_id is None:
    raise ValueError(
        "ASSET_ID could not be resolved. "
        "Set CAPSTONE_ASSET_ID, CONFIG['dataset']['asset_id'], or default_fallbacks.asset_id."
    )


DATASET_ID: str = raw_dataset_id
RUN_ID: str = raw_run_id
ASSET_ID: str = raw_asset_id


print(f"SQL schema: {CAPSTONE_SCHEMA}")
print(f"Dataset ID: {DATASET_ID}")
print(f"Run ID: {RUN_ID}")
print(f"Asset ID: {ASSET_ID}")
print(f"Synthetic run: {is_synthetic_run}")

SQL schema: capstone
Dataset ID: pump
Run ID: run__001
Asset ID: asset__001
Synthetic run: False


In [10]:

# =============================================================================
# SQL Smoke Check
# Purpose:
#   Confirm the database connection and expected capstone schemas/tables exist.
# =============================================================================

sql_smoke_check_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        table_schema,
        table_name
    FROM information_schema.tables
    WHERE table_schema IN (:capstone_schema, 'bronze', 'silver', 'gold', 'metadata')
    ORDER BY table_schema, table_name
    """,
    params={"capstone_schema": CAPSTONE_SCHEMA},
)

display(sql_smoke_check_dataframe)

,table_schema,table_name
0,bronze,sensor_observations
1,capstone,bronze_observations_input_stage
2,capstone,data_quality_events
3,capstone,pipeline_artifacts
4,capstone,pipeline_runs
5,capstone,simulation_state_control
6,capstone,simulation_timing_config
7,capstone,synthetic_observations_premelt_stage
8,capstone,synthetic_observations_timestamped_stage
9,capstone,synthetic_pump_stream


----

## Start Logging for the Cascade Modeling Stage

Before I load the modeling inputs, I want logging turned on so this notebook records what happened during the run.

This helps with debugging, traceability, and basic pipeline discipline. If I need to check what inputs were used or where a step failed, the log gives me a cleaner record than notebook output alone.

In [11]:
log_layer_paths(paths, current_layer=CONTEXT_LAYER, logger=logger)

logger.info(
    "Project paths logged for %s layer",
    CONTEXT_LAYER,
    extra={"stage": CONTEXT_STAGE, "layer": CONTEXT_LAYER},
)

ledger.add(
    kind="step",
    step=f"{CONTEXT_LAYER}_paths_logged",
    message="Logged project layer paths.",
    data={
        "stage": CONTEXT_STAGE,
        "layer": CONTEXT_LAYER,
        "log_path": str(LOG_PATH),
    },
    logger=logger,
)


2026-06-16 18:10:20,190 | INFO | capstone.gold.cascade.defaults | Project Root Path Loaded: /workspace
2026-06-16 18:10:20,197 | INFO | capstone.gold.cascade.defaults | Project Logging Path Loaded: /workspace/logs
2026-06-16 18:10:20,200 | INFO | capstone.gold.cascade.defaults | Project Artifacts Path Loaded: /workspace/artifacts
2026-06-16 18:10:20,204 | INFO | capstone.gold.cascade.defaults | Project Notebooks Path Loaded: /workspace/notebooks
2026-06-16 18:10:20,207 | INFO | capstone.gold.cascade.defaults | Project Truths Path Loaded: /workspace/artifacts/truths
2026-06-16 18:10:20,210 | INFO | capstone.gold.cascade.defaults | Project Data Path Loaded: /workspace/data
2026-06-16 18:10:20,213 | INFO | capstone.gold.cascade.defaults | Previous Layer (Silver) Path Loaded: /workspace/data/silver
2026-06-16 18:10:20,215 | INFO | capstone.gold.cascade.defaults | Previous Layer (Silver) Training Path Loaded: /workspace/data/silver/train
2026-06-16 18:10:20,218 | INFO | capstone.gold.cascad

{'ts_utc': '2026-06-16T18:10:20.228514+00:00',
 'stage': 'gold_cascade',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'gold_paths_logged',
 'message': 'Logged project layer paths.',
 'why': None,
 'consequence': None,
 'data': {'stage': 'gold_cascade',
  'layer': 'gold',
  'log_path': '/workspace/logs/gold_modeling_cascade_defaults.log'}}

In [12]:
""" 
# Original Logging Setup

# Create gold log path 
gold_log_path = paths.logs / "gold_modeling_cascade_defaults.log"

# Initial Logger
configure_logging(
    "capstone",
    gold_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.gold")

# Log load and initiation
logger.info("Gold Modeling stage starting")

# Log paths loads
log_layer_paths(paths, current_layer="gold", logger=logger)
 """

' \n# Original Logging Setup\n\n# Create gold log path \ngold_log_path = paths.logs / "gold_modeling_cascade_defaults.log"\n\n# Initial Logger\nconfigure_logging(\n    "capstone",\n    gold_log_path,\n    level=logging.DEBUG,\n    overwrite_handlers=True,\n)\n\n# Initiate Logger and log file\nlogger = logging.getLogger("capstone.gold")\n\n# Log load and initiation\nlogger.info("Gold Modeling stage starting")\n\n# Log paths loads\nlog_layer_paths(paths, current_layer="gold", logger=logger)\n '

----

## Initialize Experiment Tracking

This step starts the Weights & Biases run for the cascade modeling stage.

I am using this mainly for run tracking and artifact registration. It helps document the cascade configuration, inputs, and saved outputs, but it does not change the underlying cascade logic itself.

In [13]:
# W&B

wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=WANDB_RUN_NAME,
    job_type="gold_modeling_cascade",
    config={
        "gold_version": GOLD_VERSION,
        "dataset": DATASET_NAME,
        "stage": STAGE,
        "stage1_threshold_percentile": STAGE1_THRESHOLD_PERCENTILE,
        "stage2_selection_mode": STAGE2_SELECTION_MODE,
        "stage2_fixed_threshold_percentile": STAGE2_FIXED_THRESHOLD_PERCENTILE,
        "stage3_min_primary_sensor_hits": STAGE3_MIN_PRIMARY_SENSOR_HITS,
        "stage3_min_secondary_sensor_hits": STAGE3_MIN_SECONDARY_SENSOR_HITS,
        "stage3_rolling_window_size": STAGE3_ROLLING_WINDOW_SIZE,
        "stage3_minimum_flags_in_window": STAGE3_MINIMUM_FLAGS_IN_WINDOW,
        "gold_fit_path": str(GOLD_FIT_DATA_PATH),
        "gold_scored_path": str(GOLD_PREPROCESSED_SCALED_DATA_PATH),
        "stage1_features_path": str(STAGE1_FEATURES_PATH),
        "stage2_features_path": str(STAGE2_FEATURES_PATH),
        "stage3_primary_path": str(STAGE3_PRIMARY_PATH),
        "stage3_secondary_path": str(STAGE3_SECONDARY_PATH),
    },
)
logger.info("W&B initialized: %s", wandb_run.name)


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: dcoo230 (dcoo230-western-governors-university). Use `wandb login --relogin` to force relogin


2026-06-16 18:10:27,851 | INFO | capstone.gold.cascade.defaults | W&B initialized: gold__001


----

## Initialize the Cascade Ledger

Here I create the ledger that tracks the main steps taken during the cascade notebook.

I treat the ledger as a structured record of the run. It gives me a cleaner summary of the workflow than relying only on printed notebook output, especially when I need to review or compare cascade runs later.

In [14]:
# Ledger was initialized by load_notebook_context().
# Keep this cell as a visible notebook checkpoint instead of re-creating Ledger.

ledger.add(
    kind="step",
    step="ledger_context_ready",
    message="Ledger is available from shared notebook context.",
    data={
        "stage": CONTEXT_STAGE,
        "recipe_id": CONTEXT_RECIPE_ID,
        "log_path": str(LOG_PATH),
    },
    logger=logger,
)

logger.info(
    "Ledger ready from notebook context",
    extra={"stage": CONTEXT_STAGE, "recipe_id": CONTEXT_RECIPE_ID},
)


2026-06-16 18:10:28,474 | INFO | capstone.gold.cascade.defaults | LEDGER | {'ts_utc': '2026-06-16T18:10:28.473868+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'ledger_context_ready', 'message': 'Ledger is available from shared notebook context.', 'why': None, 'consequence': None, 'data': {'stage': 'gold_cascade', 'recipe_id': 'gold_modeling__v001_cascade', 'log_path': '/workspace/logs/gold_modeling_cascade_defaults.log'}}
2026-06-16 18:10:28,493 | INFO | capstone.gold.cascade.defaults | Ledger ready from notebook context


In [15]:
""" 
# Original Ledger Setup

ledger = Ledger(stage=STAGE, recipe_id=RECIPE_ID)

ledger.add(
    kind="step",
    step="init",
    message="Initialized ledger",
    logger=logger
)
 """

' \n# Original Ledger Setup\n\nledger = Ledger(stage=STAGE, recipe_id=RECIPE_ID)\n\nledger.add(\n    kind="step",\n    step="init",\n    message="Initialized ledger",\n    logger=logger\n)\n '

----

## Load the Gold Modeling Inputs and Resolve the Parent Truth

This is the point where I load the Gold preprocessing outputs that the cascade model depends on.

In this step I am:
- loading the scaled Gold dataframe
- resolving the parent Gold truth record
- confirming the dataset identity
- updating artifact paths from the truth record when available
- loading the Gold fit, train, and test dataframes
- loading the Stage 1, Stage 2, and Stage 3 feature artifacts

This matters because the cascade notebook should inherit the prepared Gold artifacts rather than rebuilding those inputs on its own.

In [16]:
logger.info("Loading Gold Preprocessed parquet: %s", GOLD_PREPROCESSED_SCALED_DATA_PATH)
gold_preprocessed_scaled_dataframe = load_data(GOLD_PREPROCESSED_SCALED_DATA_PATH)

GOLD_DATASET_NAME = (
    gold_preprocessed_scaled_dataframe["meta__dataset"]
    .dropna()
    .astype("string")
    .str.strip()
)
GOLD_DATASET_NAME = GOLD_DATASET_NAME[GOLD_DATASET_NAME != ""]

if len(GOLD_DATASET_NAME) == 0:
    raise ValueError("Gold cascade input dataframe is missing usable meta__dataset values.")

GOLD_DATASET_NAME = str(GOLD_DATASET_NAME.iloc[0]).strip()

gold_truth = load_parent_truth_record_from_dataframe(
    dataframe=gold_preprocessed_scaled_dataframe,
    truth_dir=TRUTHS_PATH,
    parent_layer_name="gold",
    dataset_name=GOLD_DATASET_NAME,
    column_name="meta__truth_hash",
)

DATASET_NAME = get_dataset_name_from_truth(gold_truth)
GOLD_PARENT_TRUTH_HASH = get_truth_hash(gold_truth)

PARENT_PIPELINE_MODE = get_pipeline_mode_from_truth(gold_truth)
if PARENT_PIPELINE_MODE is not None:
    PIPELINE_MODE = PARENT_PIPELINE_MODE

GOLD_TRUTH_PATH = (
    TRUTHS_PATH
    / "gold"
    / f"{DATASET_NAME}__gold__truth__{GOLD_PARENT_TRUTH_HASH}.json"
)

gold_truth_runtime_facts = gold_truth.get("runtime_facts", {})
gold_truth_artifact_paths = gold_truth.get("artifact_paths", {})

GOLD_PREPROCESSED_DATA_PATH = Path(gold_truth_artifact_paths.get("gold_preprocessed_path", str(GOLD_PREPROCESSED_DATA_PATH)))
GOLD_FIT_DATA_PATH = Path(gold_truth_artifact_paths.get("gold_fit_path", str(GOLD_FIT_DATA_PATH)))
GOLD_TEST_DATA_PATH = Path(gold_truth_artifact_paths.get("gold_test_path", str(GOLD_TEST_DATA_PATH)))
GOLD_TRAIN_DATA_PATH = Path(gold_truth_artifact_paths.get("gold_train_path", str(GOLD_TRAIN_DATA_PATH)))
STAGE1_FEATURES_PATH = Path(gold_truth_artifact_paths.get("stage1_features_path", str(STAGE1_FEATURES_PATH)))
STAGE2_FEATURES_PATH = Path(gold_truth_artifact_paths.get("stage2_features_path", str(STAGE2_FEATURES_PATH)))
STAGE3_PRIMARY_PATH = Path(gold_truth_artifact_paths.get("stage3_primary_path", str(STAGE3_PRIMARY_PATH)))
STAGE3_SECONDARY_PATH = Path(gold_truth_artifact_paths.get("stage3_secondary_path", str(STAGE3_SECONDARY_PATH)))

logger.info("Resolved Gold cascade dataset name from Gold truth: %s", DATASET_NAME)
logger.info("Resolved Gold truth path: %s", GOLD_TRUTH_PATH)

print("Gold cascade dataset name from parent truth:", DATASET_NAME)
print("Gold cascade parent truth hash:", GOLD_PARENT_TRUTH_HASH)

logger.info("Loading Gold Preprocessed parquet: %s", GOLD_PREPROCESSED_DATA_PATH)
gold_preprocessed_dataframe = load_data(GOLD_PREPROCESSED_DATA_PATH)

logger.info("Loading Gold fit parquet: %s", GOLD_FIT_DATA_PATH)
gold_fit_dataframe = load_data(GOLD_FIT_DATA_PATH)

logger.info("Loading Gold test parquet: %s", GOLD_TEST_DATA_PATH)
gold_test_dataframe = load_data(GOLD_TEST_DATA_PATH)

logger.info("Loading Gold train parquet: %s", GOLD_TRAIN_DATA_PATH)
gold_train_dataframe = load_data(GOLD_TRAIN_DATA_PATH)

logger.info("Loading Stage 1 features: %s", STAGE1_FEATURES_PATH)
#stage1_feature_columns = load_json(STAGE1_FEATURES_PATH)
stage1_feature_columns: list[str] = require_str_list(
    load_json(STAGE1_FEATURES_PATH),
    "stage1_feature_columns",
)

logger.info("Loading Stage 2 features: %s", STAGE2_FEATURES_PATH)
#stage2_feature_columns = load_json(STAGE2_FEATURES_PATH)
stage2_feature_columns: list[str] = require_str_list(
    load_json(STAGE2_FEATURES_PATH),
    "stage2_feature_columns",
)

logger.info("Loading Stage 3 primary rule sensors: %s", STAGE3_PRIMARY_PATH)
#stage3_primary_rule_sensors = load_json(STAGE3_PRIMARY_PATH)
stage3_primary_rule_sensors: list[str] = require_str_list(
    load_json(STAGE3_PRIMARY_PATH),
    "stage3_primary_rule_sensors",
)
#primary_profile_feature_columns: list[str] = require_str_list(
#    load_json(PRIMARY_PROFILE_FEATURES_PATH),
 #   "primary_profile_feature_columns",
#)

logger.info("Loading Stage 3 secondary rule sensors: %s", STAGE3_SECONDARY_PATH)
#stage3_secondary_rule_sensors = load_json(STAGE3_SECONDARY_PATH)
stage3_secondary_rule_sensors: list[str] = require_str_list(
    load_json(STAGE3_SECONDARY_PATH),
    "stage3_secondary_rule_sensors",
)
#secondary_profile_feature_columns: list[str] = require_str_list(
#    load_json(SECONDARY_PROFILE_FEATURES_PATH),
#    "secondary_profile_feature_columns",
#)

ledger.add(
    kind="step",
    step="load_modeling_inputs",
    message="Loaded Gold scaled parquet, loaded Gold truth, substituted truth-linked artifact paths, then loaded cascade inputs.",
    data={
        "gold_scaled_path": str(GOLD_PREPROCESSED_SCALED_DATA_PATH),
        "gold_truth_hash": GOLD_PARENT_TRUTH_HASH,
        "gold_truth_path": str(GOLD_TRUTH_PATH),
        "gold_preprocessed_path": str(GOLD_PREPROCESSED_DATA_PATH),
        "gold_fit_path": str(GOLD_FIT_DATA_PATH),
        "gold_test_path": str(GOLD_TEST_DATA_PATH),
        "gold_train_path": str(GOLD_TRAIN_DATA_PATH),
        "stage1_features_path": str(STAGE1_FEATURES_PATH),
        "stage2_features_path": str(STAGE2_FEATURES_PATH),
        "stage3_primary_path": str(STAGE3_PRIMARY_PATH),
        "stage3_secondary_path": str(STAGE3_SECONDARY_PATH),
        "gold_preprocessed_shape": list(gold_preprocessed_dataframe.shape),
        "gold_scaled_shape": list(gold_preprocessed_scaled_dataframe.shape),
        "gold_fit_shape": list(gold_fit_dataframe.shape),
        "gold_test_shape": list(gold_test_dataframe.shape),
        "gold_train_shape": list(gold_train_dataframe.shape),
        "stage1_feature_count": int(len(stage1_feature_columns)),
        "stage2_feature_count": int(len(stage2_feature_columns)),
        "stage3_primary_count": int(len(stage3_primary_rule_sensors)),
        "stage3_secondary_count": int(len(stage3_secondary_rule_sensors)),
    },
    logger=logger,
)

display(gold_test_dataframe.head(3))

2026-06-16 18:10:29,526 | INFO | capstone.gold.cascade.defaults | Loading Gold Preprocessed parquet: /workspace/data/gold/pump__gold__preprocessed_scaled.parquet
2026-06-16 18:10:29,531 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__preprocessed_scaled.parquet


2026-06-16 18:10:32,492 | INFO | capstone.gold.cascade.defaults | Resolved Gold cascade dataset name from Gold truth: pump
2026-06-16 18:10:32,494 | INFO | capstone.gold.cascade.defaults | Resolved Gold truth path: /workspace/artifacts/truths/gold/pump__gold__truth__1b39755619d21614721b74a063b90aceddebaa1f4b857393db937f146eccbec5.json
2026-06-16 18:10:32,499 | INFO | capstone.gold.cascade.defaults | Loading Gold Preprocessed parquet: /workspace/data/gold/pump__gold__preprocessed_prescaled.parquet
2026-06-16 18:10:32,501 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__preprocessed_prescaled.parquet


Gold cascade dataset name from parent truth: pump
Gold cascade parent truth hash: 1b39755619d21614721b74a063b90aceddebaa1f4b857393db937f146eccbec5


2026-06-16 18:10:34,904 | INFO | capstone.gold.cascade.defaults | Loading Gold fit parquet: /workspace/data/gold/pump__gold__fit_normal_only.parquet
2026-06-16 18:10:34,907 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__fit_normal_only.parquet
2026-06-16 18:10:35,766 | INFO | capstone.gold.cascade.defaults | Loading Gold test parquet: /workspace/data/gold/pump__gold__test.parquet
2026-06-16 18:10:35,773 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__test.parquet
2026-06-16 18:10:36,626 | INFO | capstone.gold.cascade.defaults | Loading Gold train parquet: /workspace/data/gold/pump__gold__train.parquet
2026-06-16 18:10:36,630 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__train.parquet
2026-06-16 18:10:38,409 | INFO | capstone.gold.cascade.defaults | Loading Stage 1 features: /workspace/artifacts/gold/pump/preprocessing/features/pump__gold__stage1_features.json
2026-06-16 18:10:38,437 | INFO | ca

,meta__asset_id,meta__dataset,meta__episode_id,meta__event_id,meta__ingested_at_utc,meta__parent_truth_hash,meta__pipeline_mode,meta__record_id,meta__run_id,meta__source_file,meta__source_row_id,meta__split,meta__truth_hash,event_time,event_step,time_index,event_date,anomaly_flag,is_anomaly,is_normal,status_normal_value,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,...,sensor_43__value_abnormal_flag,sensor_43__delta_abnormal_flag,sensor_43__any_abnormal_flag,sensor_44__value_deviation,sensor_44__delta_deviation,sensor_44__value_abnormal_flag,sensor_44__delta_abnormal_flag,sensor_44__any_abnormal_flag,sensor_45__value_deviation,sensor_45__delta_deviation,sensor_45__value_abnormal_flag,sensor_45__delta_abnormal_flag,sensor_45__any_abnormal_flag,sensor_46__value_deviation,sensor_46__delta_deviation,sensor_46__value_abnormal_flag,sensor_46__delta_abnormal_flag,sensor_46__any_abnormal_flag,sensor_47__value_deviation,sensor_47__delta_deviation,sensor_47__value_abnormal_flag,sensor_47__delta_abnormal_flag,sensor_47__any_abnormal_flag,sensor_48__value_deviation,sensor_48__delta_deviation,sensor_48__value_abnormal_flag,sensor_48__delta_abnormal_flag,sensor_48__any_abnormal_flag,sensor_49__value_deviation,sensor_49__delta_deviation,sensor_49__value_abnormal_flag,sensor_49__delta_abnormal_flag,sensor_49__any_abnormal_flag,sensor_51__value_deviation,sensor_51__delta_deviation,sensor_51__value_abnormal_flag,sensor_51__delta_abnormal_flag,sensor_51__any_abnormal_flag,normal_value_abnormal_sensor_count,normal_delta_abnormal_sensor_count,normal_total_abnormal_sensor_count,normal_training_quality_class,is_clean_normal_for_training,final_row_quality_class,row_is_clean_normal,row_is_suspect_normal,row_is_exclude_from_normal_training,machine_status__profiled,meta__row_id,meta__is_train_flag
0,asset__001,pump,5,pump:asset__001:run__001:136431,2026-06-14 20:37:14.939771+00:00,6d78344b915d1f5c31dac560d155433f966c3cfca60380...,batch,9701747606095593835,run__001,sensor.csv,136431,test,1b39755619d21614721b74a063b90aceddebaa1f4b8573...,2018-07-04 17:51:00+00:00,136431,136431,2018-07-04 00:00:00+00:00,0,0,1,NORMAL,-43.760725,-2.652173,-5.732155,-1.192304,-53.278556,-9.774008,-6.543561,-7.343809,-6.298183,-22.668203,-7.049292,-5.030354,-5.814053,-1.092581,0.009936,-0.247708,0.120209,0.028814,-2.195752,-0.353799,0.251724,0.801197,4.881352,0.050529,0.596592,0.254524,1.055766,0.386601,-2.744446,...,False,False,False,1.25641,NaN,False,False,False,1.175675,NaN,False,False,False,1.176471,NaN,False,False,False,1.391304,NaN,False,False,False,1.760174,NaN,False,False,False,3.170731,NaN,False,False,False,5.428565,NaN,False,False,False,13,0,13,suspect,False,suspect,False,True,True,normal_suspect,136431,False
1,asset__001,pump,5,pump:asset__001:run__001:136432,2026-06-14 20:37:14.939771+00:00,6d78344b915d1f5c31dac560d155433f966c3cfca60380...,batch,4242984989007806471,run__001,sensor.csv,136432,test,1b39755619d21614721b74a063b90aceddebaa1f4b8573...,2018-07-04 17:52:00+00:00,136432,136432,2018-07-04 00:00:00+00:00,0,0,1,NORMAL,-43.760726,-2.456519,-5.732155,-1.230767,-53.085787,-9.836153,-6.652261,-7.453193,-6.298183,-22.668203,-7.076084,-5.030354,-5.814053,-1.092581,0.518691,-1.254584,0.560349,0.546779,-0.798474,-0.303199,0.337026,0.834646,4.842846,0.023142,0.661646,0.255902,0.713836,0.563540,-2.522376,...,True,False,True,1.25641,0.0,False,False,False,1.283782,0.399997,False,False,False,1.176470,0.000002,False,False,False,1.391304,0.0,False,False,False,1.760174,2.111239e-07,False,False,False,3.170731,0.0,False,False,False,5.428565,0.0,False,False,False,14,0,14,suspect,False,suspect,False,True,True,normal_suspect,136432,False
2,asset__001,pump,5,pump:asset__001:run__001:136433,2026-06-14 20:37:14.939771+00:00,6d78344b915d1f5c31da

----

In [17]:
gold_preprocessed_scaled_dataframe = ensure_stable_row_id(
    gold_preprocessed_scaled_dataframe,
    row_id_column="meta__row_id",
)

ledger.add(
    kind="step",
    step="validate_cascade_row_tracking",
    message="Validated stable row identity on Gold cascade modeling input dataframe.",
    data={
        "row_id_column": "meta__row_id",
        "row_count": int(len(gold_preprocessed_scaled_dataframe)),
        "row_id_unique": bool(gold_preprocessed_scaled_dataframe["meta__row_id"].is_unique),
    },
    logger=logger,
)

2026-06-16 18:10:39,761 | INFO | capstone.gold.cascade.defaults | LEDGER | {'ts_utc': '2026-06-16T18:10:39.761146+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'validate_cascade_row_tracking', 'message': 'Validated stable row identity on Gold cascade modeling input dataframe.', 'why': None, 'consequence': None, 'data': {'row_id_column': 'meta__row_id', 'row_count': 220320, 'row_id_unique': True}}


{'ts_utc': '2026-06-16T18:10:39.761146+00:00',
 'stage': 'gold_cascade',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'validate_cascade_row_tracking',
 'message': 'Validated stable row identity on Gold cascade modeling input dataframe.',
 'why': None,
 'consequence': None,
 'data': {'row_id_column': 'meta__row_id',
  'row_count': 220320,
  'row_id_unique': True}}

----

## Rebuild the Train and Test Masks

Before fitting or evaluating the cascade, I recover the train/test split that was already stamped during Gold preprocessing.

The key point here is that this notebook is **not** creating a new split. It is reusing the split created upstream so the baseline notebook, this cascade notebook, and the comparison notebook all work from the same row partition.

In [18]:
if "meta__is_train_flag" not in gold_preprocessed_scaled_dataframe.columns:
    raise ValueError(
        "meta__is_train_flag missing from gold_preprocessed_scaled_dataframe."
    )

train_mask: pd.Series = (
    gold_preprocessed_scaled_dataframe["meta__is_train_flag"]
    .fillna(False)
    .astype(bool)
)

test_mask: pd.Series = (~train_mask).astype(bool)

train_mask_array: np.ndarray = train_mask.to_numpy(dtype=bool)
test_mask_array: np.ndarray = test_mask.to_numpy(dtype=bool)

test_labels: np.ndarray | None = None

if "anomaly_flag" in gold_preprocessed_scaled_dataframe.columns:
    test_labels = (
        gold_preprocessed_scaled_dataframe.loc[test_mask, "anomaly_flag"]
        .fillna(0)
        .astype(int)
        .to_numpy(dtype=int)
    )

logger.info("Split counts: all=%d train=%d test=%d", len(train_mask), int(train_mask.sum()), int(test_mask.sum()))

2026-06-16 18:10:40,133 | INFO | capstone.gold.cascade.defaults | Split counts: all=220320 train=136431 test=83889


### Ask

Why does the cascade notebook reuse the saved split instead of building a fresh one here?

### Answer

I want the evaluation setup to stay consistent across the Gold modeling notebooks.

If the baseline and cascade notebooks each made their own split, then the results would be harder to compare fairly. By reusing the saved Gold split, I know both modeling approaches are being judged against the same training and test boundary.

So this step is mainly about consistency and comparability.

----

## Define the Stage 3 Reference Profile Logic

Before Stage 3 can confirm alerts, I need a reference profile that describes what normal feature behavior looks like.

This helper function builds that profile from the Gold fit dataframe by summarizing each selected feature using values like:
- median
- mean
- standard deviation
- lower bound
- upper bound

The main idea is that Stage 3 should not just look for model flags. It should also compare suspicious rows against a stored picture of normal behavior.

In [19]:
def build_reference_profile(
    dataframe: pd.DataFrame,
    *,
    feature_columns: list[str],
) -> pd.DataFrame:
    reference_rows: list[dict] = []

    for feature_name in feature_columns:
        feature_series = dataframe[feature_name]

        reference_rows.append({
            "feature_name": feature_name,
            "median_value": float(feature_series.median()),
            "mean_value": float(feature_series.mean()),
            "standard_deviation": float(feature_series.std()) if not pd.isna(feature_series.std()) else 0.0,
            "lower_bound": float(feature_series.quantile(0.05)),
            "upper_bound": float(feature_series.quantile(0.95)),
        })

    reference_profile = pd.DataFrame(reference_rows)
    return reference_profile

## Build the Stage 3 Reference Profile

Here I create the actual reference profile used by Stage 3 confirmation logic.

The feature set for this profile combines:
- the broader Stage 1 modeling features
- the Stage 3 primary rule sensors
- the Stage 3 secondary rule sensors

That gives Stage 3 a normal-behavior reference it can use when checking for profile breaches and rule evidence.

In [20]:
reference_profile_features = list(dict.fromkeys(
    stage1_feature_columns + stage3_primary_rule_sensors + stage3_secondary_rule_sensors
))

reference_profile = build_reference_profile(
    gold_fit_dataframe,
    feature_columns=reference_profile_features,
)

ledger.add(
    kind="step",
    step="build_reference_profile",
    message="Built fit-period reference profile for Stage 3 confirmation logic.",
    data={
        "training_rows": int(len(gold_fit_dataframe)),
        "reference_feature_count": int(len(reference_profile_features)),
    },
    logger=logger,
)

display(reference_profile.head(10))

2026-06-16 18:10:41,552 | INFO | capstone.gold.cascade.defaults | LEDGER | {'ts_utc': '2026-06-16T18:10:41.552739+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'build_reference_profile', 'message': 'Built fit-period reference profile for Stage 3 confirmation logic.', 'why': None, 'consequence': None, 'data': {'training_rows': 77395, 'reference_feature_count': 50}}


,feature_name,median_value,mean_value,standard_deviation,lower_bound,upper_bound
0,sensor_00,0.0,-0.225564,1.983531,-2.540039,1.100022
1,sensor_01,0.0,0.021708,0.841543,-1.347824,1.456523
2,sensor_02,0.0,-0.011922,0.672431,-1.089288,1.017860
3,sensor_03,0.0,0.060602,0.691585,-0.980768,1.249997
4,sensor_04,0.0,-0.232387,1.951311,-1.927685,1.228905
5,sensor_05,0.0,0.015411,0.868819,-1.361052,1.536601
6,sensor_06,0.0,-0.038234,1.160401,-1.434801,1.804388
7,sensor_07,0.0,-0.164475,0.692269,-1.046894,1.156255
8,sensor_08,0.0,0.147561,0.763275,-0.929813,1.333325
9,sensor_09,0.0,-0.556567,3.136236,-2.666912,0.733364


----

## Quick Stage 2 Feature Review

### Ask

Why print the Stage 2 feature list?

### Answer

This is a quick inspection step to confirm that the narrower Stage 2 feature set loaded correctly from Gold preprocessing.

Since Stage 2 is supposed to be more focused than Stage 1, I want to verify the feature list before fitting the second Isolation Forest.

In [21]:
print(stage2_feature_columns)

['sensor_36', 'sensor_34', 'sensor_26', 'sensor_35', 'sensor_33', 'sensor_02', 'sensor_03', 'sensor_07', 'sensor_42', 'sensor_37', 'sensor_11', 'sensor_22']


## Prepare the Feature Matrices and Evaluation Labels

Now I build the actual feature matrices that the cascade will use.

This includes:
- Stage 1 fit features from the Gold fit dataframe
- Stage 2 fit features from the Gold fit dataframe
- Stage 1 full-dataset scoring features
- Stage 2 full-dataset scoring features
- test labels, when `anomaly_flag` is available

A detail that matters here is that the model fitting happens on the **normal-only Gold fit subset**, while scoring happens across the **full scaled Gold dataset**.

In [22]:
'''

# Deprecated feature-matrix block removed.
# The active typed feature-matrix construction is handled in the following cell.


# =========================================================
# Build model feature matrices
# =========================================================
# Keep these as DataFrames, not NumPy arrays.
# This prevents the sklearn warning:
# "X has feature names, but IsolationForest was fitted without feature names"
#
# Labels can still use .values/.to_numpy().
# The issue is only with the feature matrix passed into IsolationForest.

# Fit features from normal-only fit parquet
stage1_train_fit_features = gold_fit_dataframe.loc[:, stage1_feature_columns].copy()
stage2_train_fit_features = gold_fit_dataframe.loc[:, stage2_feature_columns].copy()

# Score features from the full scaled dataset, all rows
stage1_all_features = gold_preprocessed_scaled_dataframe.loc[:, stage1_feature_columns].copy()
stage2_all_features = gold_preprocessed_scaled_dataframe.loc[:, stage2_feature_columns].copy()


test_labels = None

if "anomaly_flag" in gold_preprocessed_scaled_dataframe.columns:
    test_labels = (
        gold_preprocessed_scaled_dataframe
        .loc[test_mask, "anomaly_flag"]
        .fillna(0)
        .astype(int)
        .values
    )
'''

'\n\n# Deprecated feature-matrix block removed.\n# The active typed feature-matrix construction is handled in the following cell.\n\n\n# =========================================================\n# Build model feature matrices\n# =========================================================\n# Keep these as DataFrames, not NumPy arrays.\n# This prevents the sklearn warning:\n# "X has feature names, but IsolationForest was fitted without feature names"\n#\n# Labels can still use .values/.to_numpy().\n# The issue is only with the feature matrix passed into IsolationForest.\n\n# Fit features from normal-only fit parquet\nstage1_train_fit_features = gold_fit_dataframe.loc[:, stage1_feature_columns].copy()\nstage2_train_fit_features = gold_fit_dataframe.loc[:, stage2_feature_columns].copy()\n\n# Score features from the full scaled dataset, all rows\nstage1_all_features = gold_preprocessed_scaled_dataframe.loc[:, stage1_feature_columns].copy()\nstage2_all_features = gold_preprocessed_scaled_data

#### Run validation guardrails

This cell runs guardrail checks before the notebook continues. These checks reduce the risk of carrying missing columns, invalid labels, or inconsistent row counts into later outputs.

In [23]:
# =========================================================
# Build model feature matrices
# =========================================================
# Keep these as DataFrames, not NumPy arrays.
# This prevents sklearn/Pylance from treating a single-column selection as a Series.

missing_stage1_features = [
    column_name
    for column_name in stage1_feature_columns
    if column_name not in gold_fit_dataframe.columns
]

missing_stage2_features = [
    column_name
    for column_name in stage2_feature_columns
    if column_name not in gold_fit_dataframe.columns
]

missing_stage1_scaled_features = [
    column_name
    for column_name in stage1_feature_columns
    if column_name not in gold_preprocessed_scaled_dataframe.columns
]

missing_stage2_scaled_features = [
    column_name
    for column_name in stage2_feature_columns
    if column_name not in gold_preprocessed_scaled_dataframe.columns
]

if missing_stage1_features:
    raise ValueError(
        "Stage 1 feature columns are missing from gold_fit_dataframe:\n"
        f"{missing_stage1_features[:25]}"
    )

if missing_stage2_features:
    raise ValueError(
        "Stage 2 feature columns are missing from gold_fit_dataframe:\n"
        f"{missing_stage2_features[:25]}"
    )

if missing_stage1_scaled_features:
    raise ValueError(
        "Stage 1 feature columns are missing from gold_preprocessed_scaled_dataframe:\n"
        f"{missing_stage1_scaled_features[:25]}"
    )

if missing_stage2_scaled_features:
    raise ValueError(
        "Stage 2 feature columns are missing from gold_preprocessed_scaled_dataframe:\n"
        f"{missing_stage2_scaled_features[:25]}"
    )

# Fit features from normal-only fit parquet.
stage1_train_fit_features: pd.DataFrame = gold_fit_dataframe.loc[
    :,
    stage1_feature_columns,
].copy()

stage2_train_fit_features: pd.DataFrame = gold_fit_dataframe.loc[
    :,
    stage2_feature_columns,
].copy()

# Score features from the full scaled dataset, all rows.
stage1_all_features: pd.DataFrame = gold_preprocessed_scaled_dataframe.loc[
    :,
    stage1_feature_columns,
].copy()

stage2_all_features: pd.DataFrame = gold_preprocessed_scaled_dataframe.loc[
    :,
    stage2_feature_columns,
].copy()

test_labels: np.ndarray | None = None

if "anomaly_flag" in gold_preprocessed_scaled_dataframe.columns:
    test_labels = (
        gold_preprocessed_scaled_dataframe
        .loc[test_mask, "anomaly_flag"]
        .fillna(0)
        .astype(int)
        .to_numpy(dtype=int)
    )

### Ask

Why am I fitting on the Gold fit subset but scoring the full scaled dataset?

### Answer

Because those steps are doing different jobs.

The fit subset gives the cascade its reference view of normal behavior. The full scaled dataset is what I want to score afterward so I can see where alerts appear across all rows.

So the overall logic is:
- fit the Stage 1 and Stage 2 models on the saved normal training subset
- score every row in the scaled Gold dataset
- evaluate the final cascade outcome on the held-out test rows

----

## Define Shared Scoring, Thresholding, and Evaluation Helpers

These helper functions break the cascade workflow into reusable pieces:
- computing anomaly scores from an Isolation Forest
- choosing a threshold by percentile
- evaluating predicted alerts against available labels

I like separating these pieces because it keeps the notebook easier to read and also makes the relationship between the baseline and cascade logic more obvious.

In [24]:
def compute_anomaly_scores_isolation_forest(
    model: IsolationForest,
    feature_matrix: pd.DataFrame | np.ndarray,
) -> np.ndarray:
    """
    Compute anomaly scores from a fitted IsolationForest model.

    Higher returned values mean more anomalous.

    Notes
    -----
    The feature_matrix can be a DataFrame or NumPy array, but for this
    project we prefer DataFrames so sklearn keeps feature-name tracking.
    """
    scores = model.score_samples(feature_matrix)
    anomaly_scores = -scores

    return anomaly_scores

#### Define percentile threshold selection

This cell performs the next transformation in `Define Shared Scoring, Thresholding, and Evaluation Helpers`. Review the output or logged message before relying on this result downstream.

In [25]:
'''
def choose_threshold_by_percentile(
    anomaly_scores: np.ndarray,
    percentile: float,
) -> float:
    return float(np.percentile(anomaly_scores, percentile))
'''

'\ndef choose_threshold_by_percentile(\n    anomaly_scores: np.ndarray,\n    percentile: float,\n) -> float:\n    return float(np.percentile(anomaly_scores, percentile))\n'

#### Define label-based model evaluation

This cell defines helper logic used by the surrounding `Define Shared Scoring, Thresholding, and Evaluation Helpers` section. Keeping the helper directly before its use makes the notebook assumptions visible and easier to audit.

In [26]:
def evaluate_against_labels(
    true_labels: np.ndarray,
    anomaly_scores: np.ndarray,
    threshold: float,
) -> dict[str, float | None]:
    true_labels_array: np.ndarray = np.asarray(true_labels, dtype=int)
    anomaly_scores_array: np.ndarray = np.asarray(anomaly_scores, dtype=float).reshape(-1)

    predicted_labels: np.ndarray = (
        anomaly_scores_array >= float(threshold)
    ).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels_array,
        predicted_labels,
        average="binary",
        zero_division=0,
    )

    results: dict[str, float | None] = {
        "threshold": float(threshold),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }

    if len(np.unique(true_labels_array)) == 2:
        results["roc_auc"] = float(
            roc_auc_score(true_labels_array, anomaly_scores_array)
        )
        results["pr_auc"] = float(
            average_precision_score(true_labels_array, anomaly_scores_array)
        )
    else:
        results["roc_auc"] = None
        results["pr_auc"] = None

    return results

----

## Run Stage 1: Broad Isolation Forest Screening

This is the first model stage of the cascade.

Stage 1 uses the broader feature set to act as an initial screen. The goal here is to cast a wider net and identify rows that look suspicious enough to move forward in the cascade.

In this step I:
- fit the Stage 1 Isolation Forest on the Gold fit subset
- score the fit subset and the full scaled dataset
- choose the Stage 1 threshold from the training-score distribution
- create the Stage 1 alert flags

This stage is intentionally broad. It is supposed to surface candidate alerts, not be the final decision by itself.

In [27]:
stage1_model = IsolationForest(
    n_estimators=STAGE1_ESTIMATOR_COUNT,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

stage1_model.fit(stage1_train_fit_features)

stage1_train_scores: np.ndarray = as_float_array(
    compute_anomaly_scores_isolation_forest(
        stage1_model,
        stage1_train_fit_features,
    ),
    "stage1_train_scores",
)

stage1_all_scores: np.ndarray = as_float_array(
    compute_anomaly_scores_isolation_forest(
        stage1_model,
        stage1_all_features,
    ),
    "stage1_all_scores",
)

stage1_threshold: float = choose_threshold_value(
    stage1_train_scores,
    STAGE1_THRESHOLD_PERCENTILE,
)

stage1_flags: np.ndarray = (
    stage1_all_scores >= stage1_threshold
).astype(int)

stage1_summary: dict[str, Any] = {
    "threshold_percentile": float(STAGE1_THRESHOLD_PERCENTILE),
    "threshold": float(stage1_threshold),
    "alert_count_all_rows": int(stage1_flags.sum()),
    "alert_count_test_rows": int(stage1_flags[test_mask_array].sum()),
}

ledger.add(
    kind="step",
    step="run_cascade_stage1",
    message="Ran Stage 1 broad Isolation Forest using saved Gold fit data and scored all rows of the scaled dataset.",
    data={
        "estimator_count": int(STAGE1_ESTIMATOR_COUNT),
        "threshold_percentile": float(STAGE1_THRESHOLD_PERCENTILE),
        "threshold": float(stage1_threshold),
        "feature_count": int(len(stage1_feature_columns)),
        "alert_count_all_rows": int(stage1_summary["alert_count_all_rows"]),
        "alert_count_test_rows": int(stage1_summary["alert_count_test_rows"]),
    },
    logger=logger,
)

2026-06-16 18:10:48,464 | INFO | capstone.gold.cascade.defaults | LEDGER | {'ts_utc': '2026-06-16T18:10:48.464049+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'run_cascade_stage1', 'message': 'Ran Stage 1 broad Isolation Forest using saved Gold fit data and scored all rows of the scaled dataset.', 'why': None, 'consequence': None, 'data': {'estimator_count': 200, 'threshold_percentile': 95.0, 'threshold': 0.4886482835351958, 'feature_count': 50, 'alert_count_all_rows': 103054, 'alert_count_test_rows': 43236}}


{'ts_utc': '2026-06-16T18:10:48.464049+00:00',
 'stage': 'gold_cascade',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'run_cascade_stage1',
 'message': 'Ran Stage 1 broad Isolation Forest using saved Gold fit data and scored all rows of the scaled dataset.',
 'why': None,
 'consequence': None,
 'data': {'estimator_count': 200,
  'threshold_percentile': 95.0,
  'threshold': 0.4886482835351958,
  'feature_count': 50,
  'alert_count_all_rows': 103054,
  'alert_count_test_rows': 43236}}

### Ask

What is Stage 1 really supposed to do in the cascade?

### Answer

Stage 1 is the broad screening step.

Its job is to catch rows that look suspicious enough to deserve a second look. I do not expect Stage 1 alone to be highly selective. In fact, it is normal for Stage 1 to raise more alerts than the later stages, because the later stages are there to narrow and confirm those alerts.

So when I review Stage 1, I mainly care that it is acting as a reasonable first filter rather than as the final answer.

----

In [28]:
stage1_input_df = build_stage_scoring_frame(
    dataframe=gold_preprocessed_scaled_dataframe,
    feature_columns=stage1_feature_columns,
    mask=None,
    row_id_column="meta__row_id",
)

stage1_results_df = score_isolation_forest_stage(
    stage_dataframe=stage1_input_df,
    model=stage1_model,
    feature_columns=stage1_feature_columns,
    stage_name="stage1",
    row_id_column="meta__row_id",
)

cascade_results = merge_stage_results_back(
    master_dataframe=gold_preprocessed_scaled_dataframe.copy(),
    stage_results_dataframe=stage1_results_df,
    stage_name="stage1",
    row_id_column="meta__row_id",
)

ledger.add(
    kind="step",
    step="score_stage1_with_row_tracking",
    message="Scored Stage 1 across the full cascade population and merged row-level outputs back using stable row identity.",
    data={
        "stage1_row_count": int(len(stage1_results_df)),
        "stage1_flag_count": int(stage1_results_df["stage1_flag"].sum()),
    },
    logger=logger,
)

2026-06-16 18:10:57,027 | INFO | capstone.gold.cascade.defaults | LEDGER | {'ts_utc': '2026-06-16T18:10:57.027359+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'score_stage1_with_row_tracking', 'message': 'Scored Stage 1 across the full cascade population and merged row-level outputs back using stable row identity.', 'why': None, 'consequence': None, 'data': {'stage1_row_count': 220320, 'stage1_flag_count': 91594}}


{'ts_utc': '2026-06-16T18:10:57.027359+00:00',
 'stage': 'gold_cascade',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'score_stage1_with_row_tracking',
 'message': 'Scored Stage 1 across the full cascade population and merged row-level outputs back using stable row identity.',
 'why': None,
 'consequence': None,
 'data': {'stage1_row_count': 220320, 'stage1_flag_count': 91594}}

#### Build review visualization

This cell builds a visual review artifact for interpretation and reporting. The plot is used to explain model behavior; it does not change the modeling inputs.

In [29]:
# =========================================================
# Synchronize Stage 1 threshold flags after row-tracked scoring
# =========================================================

if len(stage1_all_scores) != len(cascade_results):
    raise ValueError(
        "stage1_all_scores length does not match cascade_results length. "
        "Stage 1 row-tracked synchronization is unsafe."
    )

cascade_results["stage1_score"] = stage1_all_scores
cascade_results["stage1_threshold"] = float(stage1_threshold)
cascade_results["stage1_threshold_percentile"] = float(STAGE1_THRESHOLD_PERCENTILE)
cascade_results["stage1_flag"] = stage1_flags.astype(int)

ledger.add(
    kind="step",
    step="synchronize_stage1_threshold_flags",
    message="Synchronized Stage 1 row-tracked results with the configured percentile-threshold alert rule.",
    data={
        "stage1_threshold": float(stage1_threshold),
        "stage1_threshold_percentile": float(STAGE1_THRESHOLD_PERCENTILE),
        "stage1_alert_count_all_rows": int(cascade_results["stage1_flag"].sum()),
        "stage1_alert_count_test_rows": int(cascade_results.loc[test_mask, "stage1_flag"].sum()),
    },
    logger=logger,
)

print("Stage 1 threshold-synchronized alert count:", int(cascade_results["stage1_flag"].sum()))

2026-06-16 18:10:57,472 | INFO | capstone.gold.cascade.defaults | LEDGER | {'ts_utc': '2026-06-16T18:10:57.472556+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'synchronize_stage1_threshold_flags', 'message': 'Synchronized Stage 1 row-tracked results with the configured percentile-threshold alert rule.', 'why': None, 'consequence': None, 'data': {'stage1_threshold': 0.4886482835351958, 'stage1_threshold_percentile': 95.0, 'stage1_alert_count_all_rows': 103054, 'stage1_alert_count_test_rows': 43236}}


Stage 1 threshold-synchronized alert count: 103054


---

## Run Stage 2: Narrow Isolation Forest Confirmation

This is the second model stage of the cascade.

Stage 2 uses the narrower Stage 2 feature set and the fixed parameter branch from the cascade configuration. The goal here is to confirm a smaller subset of the Stage 1 alerts rather than scoring rows completely independently.

That is why the final Stage 2 flag is created by combining:
- the raw Stage 2 alert
- the Stage 1 alert

So a row has to pass through both model stages before it can move into Stage 3 confirmation logic.

In [30]:
stage2_model = IsolationForest(
    random_state=STAGE2_RANDOM_STATE,
    n_jobs=-1,
    **STAGE2_FIXED_PARAMS,
)

stage2_model.fit(stage2_train_fit_features)

stage2_train_scores: np.ndarray = as_float_array(
    compute_anomaly_scores_isolation_forest(
        stage2_model,
        stage2_train_fit_features,
    ),
    "stage2_train_scores",
)

stage2_all_scores: np.ndarray = as_float_array(
    compute_anomaly_scores_isolation_forest(
        stage2_model,
        stage2_all_features,
    ),
    "stage2_all_scores",
)

stage2_threshold: float = choose_threshold_value(
    stage2_train_scores,
    STAGE2_FIXED_THRESHOLD_PERCENTILE,
)

stage2_raw_flags: np.ndarray = (
    stage2_all_scores >= stage2_threshold
).astype(int)

stage2_flags: np.ndarray = (
    (stage1_flags == 1)
    & (stage2_raw_flags == 1)
).astype(int)

stage2_selected_threshold_percentile = float(STAGE2_FIXED_THRESHOLD_PERCENTILE)
stage2_best_params = dict(STAGE2_FIXED_PARAMS)

stage2_summary: dict[str, Any] = {
    "selection_mode": "fixed",
    "selected_threshold_percentile": float(stage2_selected_threshold_percentile),
    "threshold": float(stage2_threshold),
    "best_params": stage2_best_params,
    "raw_alert_count_all_rows": int(stage2_raw_flags.sum()),
    "raw_alert_count_test_rows": int(stage2_raw_flags[test_mask_array].sum()),
    "stage2_confirmed_count_all_rows": int(stage2_flags.sum()),
    "stage2_confirmed_count_test_rows": int(stage2_flags[test_mask_array].sum()),
}

ledger.add(
    kind="step",
    step="run_cascade_stage2_fixed",
    message="Ran Stage 2 fixed narrow Isolation Forest using the fixed branch from the cascade config.",
    data={
        "selection_mode": "fixed",
        "best_params": stage2_best_params,
        "selected_threshold_percentile": float(stage2_selected_threshold_percentile),
        "threshold": float(stage2_threshold),
        "feature_count": int(len(stage2_feature_columns)),
        "raw_alert_count_all_rows": int(stage2_summary["raw_alert_count_all_rows"]),
        "raw_alert_count_test_rows": int(stage2_summary["raw_alert_count_test_rows"]),
        "stage2_confirmed_count_all_rows": int(stage2_summary["stage2_confirmed_count_all_rows"]),
        "stage2_confirmed_count_test_rows": int(stage2_summary["stage2_confirmed_count_test_rows"]),
    },
    logger=logger,
)

2026-06-16 18:11:02,622 | INFO | capstone.gold.cascade.defaults | LEDGER | {'ts_utc': '2026-06-16T18:11:02.622437+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'run_cascade_stage2_fixed', 'message': 'Ran Stage 2 fixed narrow Isolation Forest using the fixed branch from the cascade config.', 'why': None, 'consequence': None, 'data': {'selection_mode': 'fixed', 'best_params': {'n_estimators': 300, 'max_samples': 'auto', 'contamination': 'auto', 'max_features': 1.0, 'bootstrap': False}, 'selected_threshold_percentile': 99.0, 'threshold': 0.5661461923746396, 'feature_count': 12, 'raw_alert_count_all_rows': 77536, 'raw_alert_count_test_rows': 32806, 'stage2_confirmed_count_all_rows': 70376, 'stage2_confirmed_count_test_rows': 25738}}


{'ts_utc': '2026-06-16T18:11:02.622437+00:00',
 'stage': 'gold_cascade',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'run_cascade_stage2_fixed',
 'message': 'Ran Stage 2 fixed narrow Isolation Forest using the fixed branch from the cascade config.',
 'why': None,
 'consequence': None,
 'data': {'selection_mode': 'fixed',
  'best_params': {'n_estimators': 300,
   'max_samples': 'auto',
   'contamination': 'auto',
   'max_features': 1.0,
   'bootstrap': False},
  'selected_threshold_percentile': 99.0,
  'threshold': 0.5661461923746396,
  'feature_count': 12,
  'raw_alert_count_all_rows': 77536,
  'raw_alert_count_test_rows': 32806,
  'stage2_confirmed_count_all_rows': 70376,
  'stage2_confirmed_count_test_rows': 25738}}

### Ask

Why does Stage 2 use both its own raw flag and the Stage 1 flag?

### Answer

Because Stage 2 is acting as a confirmation layer, not as a separate standalone model result.

The raw Stage 2 score tells me what the narrower model thinks. But the actual Stage 2 cascade flag only turns on when the row was already flagged by Stage 1 and then also confirmed by Stage 2.

That means Stage 2 is narrowing the Stage 1 alert set instead of replacing it.

----

In [31]:
stage2_candidate_mask = (
    cascade_results["stage1_flag"]
    .fillna(0)
    .astype(int)
    == 1
)

stage2_candidate_mask_array: np.ndarray = stage2_candidate_mask.to_numpy(dtype=bool)

stage2_input_df = build_stage_scoring_frame(
    dataframe=cascade_results,
    feature_columns=stage2_feature_columns,
    mask=stage2_candidate_mask,
    row_id_column="meta__row_id",
)

stage2_results_df = score_isolation_forest_stage(
    stage_dataframe=stage2_input_df,
    model=stage2_model,
    feature_columns=stage2_feature_columns,
    stage_name="stage2",
    row_id_column="meta__row_id",
)

# Rename raw helper outputs so they do not overwrite your threshold-based final stage2 flag logic.
stage2_results_df = stage2_results_df.rename(
    columns={
        "stage2_score": "stage2_model_score",
        "stage2_decision": "stage2_model_decision",
        "stage2_pred": "stage2_model_pred",
        "stage2_flag": "stage2_model_flag",
    }
)

cascade_results = cascade_results.merge(
    stage2_results_df[
        [
            "meta__row_id",
            "stage2_model_score",
            "stage2_model_decision",
            "stage2_model_pred",
            "stage2_model_flag",
        ]
    ],
    on="meta__row_id",
    how="left",
)

cascade_results["stage2_score"] = np.nan
cascade_results.loc[stage2_candidate_mask, "stage2_score"] = stage2_all_scores[
    stage2_candidate_mask_array
]

cascade_results["stage2_raw_flag"] = 0
cascade_results.loc[stage2_candidate_mask, "stage2_raw_flag"] = stage2_raw_flags[
    stage2_candidate_mask_array
]

cascade_results["stage2_flag"] = 0
cascade_results.loc[stage2_candidate_mask, "stage2_flag"] = stage2_flags[
    stage2_candidate_mask_array
]

cascade_results["stage2_raw_flag"] = (
    cascade_results["stage2_raw_flag"]
    .fillna(0)
    .astype(int)
)

cascade_results["stage2_flag"] = (
    cascade_results["stage2_flag"]
    .fillna(0)
    .astype(int)
)

ledger.add(
    kind="step",
    step="score_stage2_with_row_tracking",
    message="Scored Stage 2 candidate rows and merged row-level Stage 2 outputs back using stable row identity.",
    data={
        "stage2_candidate_count": int(stage2_candidate_mask.sum()),
        "stage2_scored_row_count": int(len(stage2_results_df)),
        "stage2_model_flag_count": int(stage2_results_df["stage2_model_flag"].sum()),
        "stage2_final_flag_count": int(cascade_results["stage2_flag"].sum()),
    },
    logger=logger,
)

2026-06-16 18:11:10,816 | INFO | capstone.gold.cascade.defaults | LEDGER | {'ts_utc': '2026-06-16T18:11:10.816101+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'score_stage2_with_row_tracking', 'message': 'Scored Stage 2 candidate rows and merged row-level Stage 2 outputs back using stable row identity.', 'why': None, 'consequence': None, 'data': {'stage2_candidate_count': 103054, 'stage2_scored_row_count': 103054, 'stage2_model_flag_count': 99335, 'stage2_final_flag_count': 70376}}


{'ts_utc': '2026-06-16T18:11:10.816101+00:00',
 'stage': 'gold_cascade',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'score_stage2_with_row_tracking',
 'message': 'Scored Stage 2 candidate rows and merged row-level Stage 2 outputs back using stable row identity.',
 'why': None,
 'consequence': None,
 'data': {'stage2_candidate_count': 103054,
  'stage2_scored_row_count': 103054,
  'stage2_model_flag_count': 99335,
  'stage2_final_flag_count': 70376}}

----

## Continue the Cascade Results Dataframe from Row-Tracked Stage Outputs

At this point the working `cascade_results` dataframe already contains the Stage 1 row-level outputs merged by stable row identity, and Stage 2 row-level outputs are now merged back in for the Stage 1 candidate rows.

So I do not rebuild the results dataframe from scratch here. Instead, I continue using the row-tracked `cascade_results` dataframe as the single source of truth for Stage 3 rule logic.

----

## Validate That the Stage 3 Rule Sensors Exist

Before I run Stage 3 confirmation logic, I want to verify that the required primary and secondary rule sensors are actually present in the scored dataframe.

This is a quick trust check. If those sensors are missing, then the Stage 3 confirmation logic would be working from an incomplete rule set, which would weaken the final cascade decision.

In [32]:
# --- Stage 3 sanity check: ensure rule sensors exist in scored dataframe
missing_primary = [column for column in stage3_primary_rule_sensors if column not in cascade_results.columns]
missing_secondary = [column for column in stage3_secondary_rule_sensors if column not in cascade_results.columns]

logger.info("Stage3 missing sensors: primary=%d secondary=%d", len(missing_primary), len(missing_secondary))

if missing_primary:
    logger.warning("Missing Stage3 PRIMARY sensors (showing up to 20): %s", missing_primary[:20])
if missing_secondary:
    logger.warning("Missing Stage3 SECONDARY sensors (showing up to 20): %s", missing_secondary[:20])


2026-06-16 18:11:11,364 | INFO | capstone.gold.cascade.defaults | Stage3 missing sensors: primary=0 secondary=0


----

## Compute the Primary Profile Breach Count

Here I apply the primary breach logic to the cascade results dataframe.

This creates a row-level count of how many Stage 3 primary sensors are outside the expected normal profile bounds. That count becomes one of the strongest pieces of Stage 3 evidence.

In [33]:

def compute_primary_breach_count(
    dataframe: pd.DataFrame,
    *,
    reference_profile: pd.DataFrame,
    feature_columns: list[str],
) -> pd.Series:
    reference_lookup = reference_profile.set_index("feature_name")[["lower_bound", "upper_bound"]]

    breach_counts = pd.Series(0, index=dataframe.index, dtype=int)

    for feature_name in feature_columns:
        if feature_name not in dataframe.columns or feature_name not in reference_lookup.index:
            continue

        lower = reference_lookup.loc[feature_name, "lower_bound"]
        upper = reference_lookup.loc[feature_name, "upper_bound"]

        breach_flag = ((dataframe[feature_name] < lower) | (dataframe[feature_name] > upper)).astype(int)
        breach_counts = breach_counts + breach_flag

    breach_counts.name = "stage3_profile_breach_count"
    return breach_counts





## Define the Secondary Corroboration Logic

This helper function does a similar job for the secondary rule sensors.

The difference is that these secondary sensors are used more as corroboration than as the main profile breach signal. In other words, they help support the alert rather than carrying the same priority as the primary sensor group.

In [34]:
cascade_results["stage3_profile_breach_count"] = compute_primary_breach_count(
    cascade_results,
    reference_profile=reference_profile,
    feature_columns=stage3_primary_rule_sensors,
)

----

## Compute the Secondary Breach Count

Here I apply the secondary breach logic to the cascade results dataframe.

This creates the row-level count of secondary rule sensors that move outside their reference profile bounds. Later this gets converted into a corroboration flag that contributes to the Stage 3 evidence count.

In [35]:
def compute_secondary_breach_count(
    dataframe: pd.DataFrame,
    *,
    reference_profile: pd.DataFrame,
    feature_columns: list[str],
) -> pd.Series:
    reference_lookup = reference_profile.set_index("feature_name").to_dict("index")
    breach_counts = pd.Series(0, index=dataframe.index, dtype=int)

    for feature_name in feature_columns:
        if feature_name not in reference_lookup:
            continue

        lower_bound = reference_lookup[feature_name]["lower_bound"]
        upper_bound = reference_lookup[feature_name]["upper_bound"]

        feature_breach_flag = (
            (dataframe[feature_name] < lower_bound) |
            (dataframe[feature_name] > upper_bound)
        ).astype(int)

        breach_counts = breach_counts + feature_breach_flag

    breach_counts.name = "stage3_secondary_breach_count"
    return breach_counts

#### Compute Stage 3 secondary breach evidence

This cell performs the next transformation in `Compute the Secondary Breach Count`. Review the output or logged message before relying on this result downstream.

In [36]:
cascade_results["stage3_secondary_breach_count"] = compute_secondary_breach_count(
    cascade_results,
    reference_profile=reference_profile,
    feature_columns=stage3_secondary_rule_sensors,
)

----

## Define the Persistence Logic

Not every abnormal point matters equally. Some are isolated spikes, while others persist across nearby rows.

This helper function checks for persistence by looking at recent Stage 2 flags inside a rolling window. If enough flags appear within that window, the row receives a persistence flag.

In [37]:
def compute_persistence_flag(
    source_flags: pd.Series,
    *,
    rolling_window_size: int = 3,
    minimum_flags_in_window: int = 2,
) -> pd.Series:
    persistence_flag = (
        source_flags
        .rolling(window=rolling_window_size, min_periods=1)
        .sum()
        .ge(minimum_flags_in_window)
        .astype(int)
    )

    persistence_flag.name = "stage3_persistence_flag"
    return persistence_flag

## Compute the Persistence Flag

Here I apply the persistence logic to the Stage 2 flags.

The goal is to capture short-run continuity. A row that belongs to a cluster of nearby Stage 2 flags has stronger evidence than a single one-off hit by itself.

In [38]:
cascade_results["stage3_persistence_flag"] = compute_persistence_flag(
    cascade_results["stage2_flag"],
    rolling_window_size=STAGE3_ROLLING_WINDOW_SIZE,
    minimum_flags_in_window=STAGE3_MINIMUM_FLAGS_IN_WINDOW,
)


----

## Define the Drift Logic

This helper function checks for rolling drift behavior in the Stage 3 watch features.

The idea is to compare each feature to its recent rolling median and flag rows where the current value drifts far enough away from that recent local behavior. This gives Stage 3 another kind of evidence that is different from simple threshold breaches.

In [39]:
def compute_drift_flag(
    dataframe: pd.DataFrame,
    *,
    feature_columns: list[str],
    rolling_window_size: int = 5,
    drift_threshold_multiplier: float = 1.0,
) -> pd.Series:
    drift_trigger_counts = pd.Series(0, index=dataframe.index, dtype=int)

    for feature_name in feature_columns:
        feature_series = dataframe[feature_name]
        feature_standard_deviation = feature_series.std()

        if pd.isna(feature_standard_deviation) or feature_standard_deviation == 0:
            continue

        rolling_median = feature_series.rolling(window=rolling_window_size, min_periods=1).median()
        rolling_delta = (feature_series - rolling_median).abs()

        feature_drift_flag = (
            rolling_delta > (feature_standard_deviation * drift_threshold_multiplier)
        ).astype(int)

        drift_trigger_counts = drift_trigger_counts + feature_drift_flag

    drift_flag = (drift_trigger_counts >= 1).astype(int)
    drift_flag.name = "stage3_drift_flag"
    return drift_flag

----

## Compute the Drift Flag

Here I apply the drift logic to the combined Stage 3 watch feature set.

This produces a row-level drift flag that tells me whether at least one watched feature is moving far enough away from its recent rolling pattern to count as drift evidence.

In [40]:
stage3_rule_watch_features = list(dict.fromkeys(
    stage3_primary_rule_sensors + stage3_secondary_rule_sensors
))

cascade_results["stage3_drift_flag"] = compute_drift_flag(
    cascade_results,
    feature_columns=stage3_rule_watch_features,
    rolling_window_size=5,
    drift_threshold_multiplier=1.0,
)

----

## Build the Final Stage 3 Evidence Flags and Final Cascade Decision

Now I combine the Stage 3 evidence into the final confirmation logic.

This section creates:
- the primary profile breach flag
- the secondary corroboration flag
- the persistence flag
- the drift flag
- the overall Stage 3 evidence count
- the final `cascade_final_flag`

The final cascade decision is intentionally strict. A row must first pass through Stage 1 and Stage 2, and then it must also show enough Stage 3 evidence to be considered a final cascade alert.

In [41]:
cascade_results["stage3_profile_breach_flag"] = (
    cascade_results["stage3_profile_breach_count"] >= STAGE3_MIN_PRIMARY_SENSOR_HITS
).astype(int)

cascade_results["stage3_corroboration_flag"] = (
    cascade_results["stage3_secondary_breach_count"] >= STAGE3_MIN_SECONDARY_SENSOR_HITS
).astype(int)

cascade_results["stage3_rule_evidence_count"] = (
    cascade_results["stage3_profile_breach_flag"] +
    cascade_results["stage3_persistence_flag"] +
    cascade_results["stage3_drift_flag"] +
    cascade_results["stage3_corroboration_flag"]
)

cascade_results["cascade_final_flag"] = (
    (cascade_results["stage1_flag"] == 1) &
    (cascade_results["stage2_flag"] == 1) &
    (
        (cascade_results["stage3_profile_breach_count"] >= STAGE3_MIN_PRIMARY_SENSOR_HITS) |
        (cascade_results["stage3_rule_evidence_count"] >= 2)
    )
).astype(int)

stage3_summary = {
    "primary_rule_sensor_count": int(len(stage3_primary_rule_sensors)),
    "secondary_rule_sensor_count": int(len(stage3_secondary_rule_sensors)),
    "profile_breach_rows_all": int((cascade_results["stage3_profile_breach_flag"] == 1).sum()),
    "profile_breach_rows_test": int(cascade_results.loc[test_mask, "stage3_profile_breach_flag"].sum()),
    "corroboration_rows_all": int((cascade_results["stage3_corroboration_flag"] == 1).sum()),
    "corroboration_rows_test": int(cascade_results.loc[test_mask, "stage3_corroboration_flag"].sum()),
    "persistence_rows_all": int((cascade_results["stage3_persistence_flag"] == 1).sum()),
    "persistence_rows_test": int(cascade_results.loc[test_mask, "stage3_persistence_flag"].sum()),
    "drift_rows_all": int((cascade_results["stage3_drift_flag"] == 1).sum()),
    "drift_rows_test": int(cascade_results.loc[test_mask, "stage3_drift_flag"].sum()),
    "final_alert_count_all_rows": int(cascade_results["cascade_final_flag"].sum()),
    "final_alert_count_test_rows": int(cascade_results.loc[test_mask, "cascade_final_flag"].sum()),
}

ledger.add(
    kind="step",
    step="run_cascade_stage3_confirmation",
    message="Applied Stage 3 confirmation checks to all rows of the scaled dataset.",
    data=stage3_summary,
    logger=logger,
)

display(cascade_results.head(5))

2026-06-16 18:11:16,873 | INFO | capstone.gold.cascade.defaults | LEDGER | {'ts_utc': '2026-06-16T18:11:16.873520+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'run_cascade_stage3_confirmation', 'message': 'Applied Stage 3 confirmation checks to all rows of the scaled dataset.', 'why': None, 'consequence': None, 'data': {'primary_rule_sensor_count': 5, 'secondary_rule_sensor_count': 8, 'profile_breach_rows_all': 92627, 'profile_breach_rows_test': 46280, 'corroboration_rows_all': 173374, 'corroboration_rows_test': 78953, 'persistence_rows_all': 70006, 'persistence_rows_test': 25398, 'drift_rows_all': 1563, 'drift_rows_test': 441, 'final_alert_count_all_rows': 69362, 'final_alert_count_test_rows': 24895}}


,meta__asset_id,meta__dataset,meta__episode_id,meta__event_id,meta__ingested_at_utc,meta__parent_truth_hash,meta__pipeline_mode,meta__record_id,meta__run_id,meta__source_file,meta__source_row_id,meta__split,meta__truth_hash,event_time,event_step,time_index,event_date,anomaly_flag,is_anomaly,is_normal,status_normal_value,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,...,sensor_47__delta_abnormal_flag,sensor_47__any_abnormal_flag,sensor_48__value_deviation,sensor_48__delta_deviation,sensor_48__value_abnormal_flag,sensor_48__delta_abnormal_flag,sensor_48__any_abnormal_flag,sensor_49__value_deviation,sensor_49__delta_deviation,sensor_49__value_abnormal_flag,sensor_49__delta_abnormal_flag,sensor_49__any_abnormal_flag,sensor_51__value_deviation,sensor_51__delta_deviation,sensor_51__value_abnormal_flag,sensor_51__delta_abnormal_flag,sensor_51__any_abnormal_flag,normal_value_abnormal_sensor_count,normal_delta_abnormal_sensor_count,normal_total_abnormal_sensor_count,normal_training_quality_class,is_clean_normal_for_training,final_row_quality_class,row_is_clean_normal,row_is_suspect_normal,row_is_exclude_from_normal_training,machine_status__profiled,meta__row_id,meta__is_train_flag,stage1_score,stage1_decision,stage1_pred,stage1_flag,stage1_threshold,stage1_threshold_percentile,stage2_model_score,stage2_model_decision,stage2_model_pred,stage2_model_flag,stage2_score,stage2_raw_flag,stage2_flag,stage3_profile_breach_count,stage3_secondary_breach_count,stage3_persistence_flag,stage3_drift_flag,stage3_profile_breach_flag,stage3_corroboration_flag,stage3_rule_evidence_count,cascade_final_flag
0,asset__001,pump,0,pump:asset__001:run__001:0,2026-06-14 20:37:14.939771+00:00,6d78344b915d1f5c31dac560d155433f966c3cfca60380...,batch,14598431322315673869,run__001,sensor.csv,0,train,1b39755619d21614721b74a063b90aceddebaa1f4b8573...,2018-04-01 00:00:00+00:00,0,0,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,0.140011,-0.695652,0.464286,1.269226,0.108436,-0.168590,-0.369557,0.124978,0.736837,-0.266728,-1.260975,0.925032,-0.034035,-0.123905,-0.461029,-0.672626,0.304680,0.142472,-0.295277,-0.461054,-0.091023,-0.172259,-0.121836,0.099551,-0.311720,-0.479814,-0.666538,-1.737734,-0.546495,...,False,False,0.716570,NaN,False,False,False,2.780487,NaN,False,False,False,0.275862,NaN,False,False,False,4,0,4,clean,True,clean,True,False,False,normal_clean,0,True,0.409368,0.090632,1,0,0.488648,95.0,NaN,NaN,NaN,NaN,NaN,0,0,1,2,0,0,0,1,1,0
1,asset__001,pump,0,pump:asset__001:run__001:1,2026-06-14 20:37:14.939771+00:00,6d78344b915d1f5c31dac560d155433f966c3cfca60380...,batch,15954729095895098000,run__001,sensor.csv,1,train,1b39755619d21614721b74a063b90aceddebaa1f4b8573...,2018-04-01 00:01:00+00:00,1,1,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,0.140011,-0.695652,0.464286,1.269226,0.108436,-0.168590,-0.369557,0.124978,0.736837,-0.266728,-1.260975,0.925032,-0.034035,-0.123905,-0.461029,-0.672626,0.304680,0.142472,-0.295277,-0.461054,-0.091023,-0.172259,-0.121836,0.099551,-0.311720,-0.479814,-0.666538,-1.737734,-0.546495,...,False,False,0.716570,0.000000,False,False,False,2.780487,0.000000,False,False,False,0.275862,0.000000,False,False,False,4,0,4,clean,True,clean,True,False,False,normal_clean,1,True,0.409368,0.090632,1,0,0.488648,95.0,NaN,NaN,NaN,NaN,NaN,0,0,1,2,0,0,0,1,1,0
2,asset__001,pump,0,pump:asset__001:run__001:2,2026-06-14 20:37:14.939771+00:00,6d78344b915d1f5c31dac560d155433f966c3cfca60380...,batch,10041703297090838359,run__001,sensor.csv,2,train,1b39755619d21614721b74a063b90aceddebaa1f4b8573...,2018-04-01 00:02:00+00:00,2,2,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,-0.280002,-0.565216,0.464286,1.307690,0.578311,-0.537436,-0.630443,-0.078128,0.859651,-0.666728,-1.160344,1.006862,0.146473,-0.108279,0.230324,-0.259305,-0.439206,-0.506023,

#### Finalize cascade stage flag columns

This cell records traceability information for the current transformation. The ledger/log output documents what changed, why it changed, and how the step should be audited.

In [42]:

cascade_results = finalize_stage_flag_columns(
    cascade_results,
    stage_names=["stage1", "stage2", "stage3"],
)

if "stage3_confirmed_flag" in cascade_results.columns and "stage3_flag" not in cascade_results.columns:
    cascade_results["stage3_flag"] = cascade_results["stage3_confirmed_flag"].fillna(0).astype(int)

ledger.add(
    kind="step",
    step="finalize_stage_flags",
    message="Finalized sparse cascade stage flags after Stage 3 rule processing.",
    data={
        "stage1_flag_count": int(cascade_results["stage1_flag"].sum()) if "stage1_flag" in cascade_results.columns else 0,
        "stage2_flag_count": int(cascade_results["stage2_flag"].sum()) if "stage2_flag" in cascade_results.columns else 0,
        "stage3_flag_count": int(cascade_results["stage3_flag"].sum()) if "stage3_flag" in cascade_results.columns else 0,
        "cascade_final_flag_count": int(cascade_results["cascade_final_flag"].sum()) if "cascade_final_flag" in cascade_results.columns else 0,
    },
    logger=logger,
)

2026-06-16 18:11:20,782 | INFO | capstone.gold.cascade.defaults | LEDGER | {'ts_utc': '2026-06-16T18:11:20.782258+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'finalize_stage_flags', 'message': 'Finalized sparse cascade stage flags after Stage 3 rule processing.', 'why': None, 'consequence': None, 'data': {'stage1_flag_count': 103054, 'stage2_flag_count': 70376, 'stage3_flag_count': 0, 'cascade_final_flag_count': 69362}}


{'ts_utc': '2026-06-16T18:11:20.782258+00:00',
 'stage': 'gold_cascade',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'finalize_stage_flags',
 'message': 'Finalized sparse cascade stage flags after Stage 3 rule processing.',
 'why': None,
 'consequence': None,
 'data': {'stage1_flag_count': 103054,
  'stage2_flag_count': 70376,
  'stage3_flag_count': 0,
  'cascade_final_flag_count': 69362}}

### Ask

How should I think about the final Stage 3 rule?

### Answer

Stage 3 is the confirmation layer that tries to reduce weak or isolated model alerts.

In this notebook, a row can become a final cascade alert in one of two main ways:
- it shows enough primary profile breaches on its own
- or it builds enough combined rule evidence across the Stage 3 checks

So Stage 3 is not replacing the model stages. It is acting as a final confirmation layer after the row has already passed through both model filters.

----

## Build the Main Cascade Metrics

Here I summarize the main cascade results.

This includes:
- Stage 1 alert counts
- Stage 2 alert counts
- final cascade alert counts
- test-row alert counts
- evaluation metrics such as precision, recall, and F1 when test labels are available

This is the main performance snapshot for the default cascade notebook.

In [43]:
cascade_metrics: dict[str, Any] = {
    "model": "3-Stage Cascade",
    "stage1_alert_count_all_rows": int(cascade_results["stage1_flag"].sum()),
    "stage2_alert_count_all_rows": int(cascade_results["stage2_flag"].sum()),
    "final_alert_count_all_rows": int(cascade_results["cascade_final_flag"].sum()),
    "stage1_alert_count_test_rows": int(cascade_results.loc[test_mask, "stage1_flag"].sum()),
    "stage2_alert_count_test_rows": int(cascade_results.loc[test_mask, "stage2_flag"].sum()),
    "final_alert_count_test_rows": int(cascade_results.loc[test_mask, "cascade_final_flag"].sum()),
}

if test_labels is not None:
    test_labels_array: np.ndarray = np.asarray(test_labels, dtype=int)

    cascade_test_flags: np.ndarray = (
        cascade_results
        .loc[test_mask, "cascade_final_flag"]
        .fillna(0)
        .astype(int)
        .to_numpy(dtype=int)
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        test_labels_array,
        cascade_test_flags,
        average="binary",
        zero_division=0,
    )

    cascade_metrics["precision"] = float(precision)
    cascade_metrics["recall"] = float(recall)
    cascade_metrics["f1"] = float(f1)

display(cascade_metrics)

{'model': '3-Stage Cascade',
 'stage1_alert_count_all_rows': 103054,
 'stage2_alert_count_all_rows': 70376,
 'final_alert_count_all_rows': 69362,
 'stage1_alert_count_test_rows': 43236,
 'stage2_alert_count_test_rows': 25738,
 'final_alert_count_test_rows': 24895,
 'precision': 0.0030929905603534844,
 'recall': 0.652542372881356,
 'f1': 0.006156798464798305}

### Ask

What should I pay attention to in the cascade metrics?

### Answer

The main things I care about are:
- how many rows survive from Stage 1 into Stage 2
- how many rows survive from Stage 2 into the final cascade output
- whether the final alert count drops in a controlled way rather than collapsing too aggressively
- precision, recall, and F1 on the test rows when labels are available

For this notebook, the key question is not just whether the cascade detects anomalies. It is whether the layered filtering helps produce a cleaner alert set than the baseline.

In [44]:
# =========================================================
# Cascade output validation on held-out test rows
# =========================================================

def validate_cascade_output(
    results_dataframe: pd.DataFrame,
    *,
    test_mask: pd.Series,
    label_column: str = "anomaly_flag",
    row_id_column: str = "meta__row_id",
    final_flag_column: str = "cascade_final_flag",
) -> dict[str, Any]:
    required_columns = [
        row_id_column,
        "meta__is_train_flag",
        "stage1_flag",
        "stage2_raw_flag",
        "stage2_flag",
        final_flag_column,
    ]

    for column_name in required_columns:
        if column_name not in results_dataframe.columns:
            raise ValueError(f"Missing required cascade output column: {column_name}")

    if len(test_mask) != len(results_dataframe):
        raise ValueError("test_mask length does not match cascade results dataframe.")

    if not results_dataframe[row_id_column].is_unique:
        raise ValueError(f"{row_id_column} is not unique in cascade results.")

    binary_columns = [
        "stage1_flag",
        "stage2_raw_flag",
        "stage2_flag",
        final_flag_column,
    ]

    for column_name in binary_columns:
        invalid_values = sorted(
            set(results_dataframe[column_name].dropna().unique()) - {0, 1}
        )

        if invalid_values:
            raise ValueError(f"{column_name} contains non-binary values: {invalid_values}")

    bad_stage2_gate_rows = results_dataframe.loc[
        (results_dataframe["stage2_flag"] == 1)
        & (results_dataframe["stage1_flag"] != 1)
    ]

    if len(bad_stage2_gate_rows) > 0:
        raise ValueError(
            "Stage 2 gate violation: stage2_flag has rows that did not pass stage1_flag. "
            f"Bad row count: {len(bad_stage2_gate_rows)}"
        )

    bad_final_gate_rows = results_dataframe.loc[
        (results_dataframe[final_flag_column] == 1)
        & (results_dataframe["stage2_flag"] != 1)
    ]

    if len(bad_final_gate_rows) > 0:
        raise ValueError(
            f"Final cascade gate violation: {final_flag_column} has rows that did not pass stage2_flag. "
            f"Bad row count: {len(bad_final_gate_rows)}"
        )

    test_dataframe = results_dataframe.loc[test_mask].copy()

    validation_summary: dict[str, Any] = {
        "all_rows": int(len(results_dataframe)),
        "test_rows": int(len(test_dataframe)),
        "stage1_alert_count_all_rows": int(results_dataframe["stage1_flag"].sum()),
        "stage1_alert_count_test_rows": int(test_dataframe["stage1_flag"].sum()),
        "stage2_raw_alert_count_all_rows": int(results_dataframe["stage2_raw_flag"].sum()),
        "stage2_raw_alert_count_test_rows": int(test_dataframe["stage2_raw_flag"].sum()),
        "stage2_alert_count_all_rows": int(results_dataframe["stage2_flag"].sum()),
        "stage2_alert_count_test_rows": int(test_dataframe["stage2_flag"].sum()),
        "final_alert_count_all_rows": int(results_dataframe[final_flag_column].sum()),
        "final_alert_count_test_rows": int(test_dataframe[final_flag_column].sum()),
    }

    if label_column in results_dataframe.columns:
        y_true: np.ndarray = (
            test_dataframe[label_column]
            .fillna(0)
            .astype(int)
            .to_numpy(dtype=int)
        )

        y_pred: np.ndarray = (
            test_dataframe[final_flag_column]
            .fillna(0)
            .astype(int)
            .to_numpy(dtype=int)
        )

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            average="binary",
            zero_division=0,
        )

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1],
        ).ravel()

        validation_summary.update(
            {
                "test_anomaly_rows": int(y_true.sum()),
                "precision": float(precision),
                "recall": float(recall),
                "f1": float(f1),
                "tn": int(tn),
                "fp": int(fp),
                "fn": int(fn),
                "tp": int(tp),
            }
        )

    print("Cascade output validation passed.")
    return validation_summary

#### Validate cascade output columns and counts

This cell runs guardrail checks before the notebook continues. These checks reduce the risk of carrying missing columns, invalid labels, or inconsistent row counts into later outputs.

In [45]:


cascade_output_validation = validate_cascade_output(
    cascade_results,
    test_mask=test_mask,
    final_flag_column="cascade_final_flag",
)

ledger.add(
    kind="step",
    step="validate_cascade_output",
    message="Validated cascade staged outputs and final held-out test metrics.",
    data=cascade_output_validation,
    logger=logger,
)

display(pd.DataFrame([cascade_output_validation]))

2026-06-16 18:11:22,600 | INFO | capstone.gold.cascade.defaults | LEDGER | {'ts_utc': '2026-06-16T18:11:22.600052+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'validate_cascade_output', 'message': 'Validated cascade staged outputs and final held-out test metrics.', 'why': None, 'consequence': None, 'data': {'all_rows': 220320, 'test_rows': 83889, 'stage1_alert_count_all_rows': 103054, 'stage1_alert_count_test_rows': 43236, 'stage2_raw_alert_count_all_rows': 70376, 'stage2_raw_alert_count_test_rows': 25738, 'stage2_alert_count_all_rows': 70376, 'stage2_alert_count_test_rows': 25738, 'final_alert_count_all_rows': 69362, 'final_alert_count_test_rows': 24895, 'test_anomaly_rows': 118, 'precision': 0.0030929905603534844, 'recall': 0.652542372881356, 'f1': 0.006156798464798305, 'tn': 58953, 'fp': 24818, 'fn': 41, 'tp': 77}}


Cascade output validation passed.


,all_rows,test_rows,stage1_alert_count_all_rows,stage1_alert_count_test_rows,stage2_raw_alert_count_all_rows,stage2_raw_alert_count_test_rows,stage2_alert_count_all_rows,stage2_alert_count_test_rows,final_alert_count_all_rows,final_alert_count_test_rows,test_anomaly_rows,precision,recall,f1,tn,fp,fn,tp
0,220320,83889,103054,43236,70376,25738,70376,25738,69362,24895,118,0.003093,0.652542,0.006157,58953,24818,41,77


----

In [46]:
cascade_results = finalize_stage_flag_columns(
    cascade_results,
    stage_names=["stage1", "stage2", "stage3"],
)

if "stage3_confirmed_flag" in cascade_results.columns and "stage3_flag" not in cascade_results.columns:
    cascade_results["stage3_flag"] = cascade_results["stage3_confirmed_flag"].fillna(0).astype(int)

ledger.add(
    kind="step",
    step="finalize_stage_flags",
    message="Finalized sparse cascade stage flags after Stage 3 rule processing.",
    data={
        "stage1_flag_count": int(cascade_results["stage1_flag"].sum()) if "stage1_flag" in cascade_results.columns else 0,
        "stage2_flag_count": int(cascade_results["stage2_flag"].sum()) if "stage2_flag" in cascade_results.columns else 0,
        "stage3_flag_count": int(cascade_results["stage3_flag"].sum()) if "stage3_flag" in cascade_results.columns else 0,
        "cascade_final_flag_count": int(cascade_results["cascade_final_flag"].sum()) if "cascade_final_flag" in cascade_results.columns else 0,
    },
    logger=logger,
)

2026-06-16 18:11:23,978 | INFO | capstone.gold.cascade.defaults | LEDGER | {'ts_utc': '2026-06-16T18:11:23.978258+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'finalize_stage_flags', 'message': 'Finalized sparse cascade stage flags after Stage 3 rule processing.', 'why': None, 'consequence': None, 'data': {'stage1_flag_count': 103054, 'stage2_flag_count': 70376, 'stage3_flag_count': 0, 'cascade_final_flag_count': 69362}}


{'ts_utc': '2026-06-16T18:11:23.978258+00:00',
 'stage': 'gold_cascade',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'finalize_stage_flags',
 'message': 'Finalized sparse cascade stage flags after Stage 3 rule processing.',
 'why': None,
 'consequence': None,
 'data': {'stage1_flag_count': 103054,
  'stage2_flag_count': 70376,
  'stage3_flag_count': 0,
  'cascade_final_flag_count': 69362}}

----

## Build the Cascade Summary, Threshold Records, and Truth Artifact

Now I convert the cascade results into formal pipeline artifacts.

This section does several important things:
- summarizes the cascade thresholds and metrics
- builds the main cascade summary record
- creates the Gold cascade truth record
- stores runtime facts and artifact paths
- links the cascade output back to the parent Gold truth
- prepares the final saved deliverables for downstream comparison

At this point, the notebook output becomes more than just a scored dataframe. It becomes a tracked modeling artifact in the pipeline.

In [47]:
stage1_alert_count_all_rows = int(cascade_results["stage1_flag"].sum())
stage2_alert_count_all_rows = int(cascade_results["stage2_flag"].sum())
final_cascade_alert_count_all_rows = int(cascade_results["cascade_final_flag"].sum())

stage1_alert_count_test_rows = int(cascade_results.loc[test_mask, "stage1_flag"].sum())
stage2_alert_count_test_rows = int(cascade_results.loc[test_mask, "stage2_flag"].sum())
final_cascade_alert_count_test_rows = int(cascade_results.loc[test_mask, "cascade_final_flag"].sum())

cascade_thresholds = {
    "cascade_variant": CASCADE_VARIANT,
    "stage1_threshold_percentile": float(STAGE1_THRESHOLD_PERCENTILE),
    "stage1_threshold": float(stage1_threshold),
    "stage2_selection_mode": STAGE2_SELECTION_MODE,
    "stage2_selected_threshold_percentile": float(stage2_selected_threshold_percentile),
    "stage2_threshold": float(stage2_threshold),
    "stage2_best_params": stage2_best_params,
}

cascade_summary = {
    "dataset_name": DATASET_NAME,
    "cascade_variant": CASCADE_VARIANT,
    "cascade_metrics": cascade_metrics,
    "stage1_alert_count_all_rows": stage1_alert_count_all_rows,
    "stage2_alert_count_all_rows": stage2_alert_count_all_rows,
    "final_cascade_alert_count_all_rows": final_cascade_alert_count_all_rows,
    "stage1_alert_count_test_rows": stage1_alert_count_test_rows,
    "stage2_alert_count_test_rows": stage2_alert_count_test_rows,
    "final_cascade_alert_count_test_rows": final_cascade_alert_count_test_rows,
    "result_row_count": int(len(cascade_results)),
    "stage1_feature_count": int(len(stage1_feature_columns)),
    "stage2_feature_count": int(len(stage2_feature_columns)),
    "stage3_primary_rule_count": int(len(stage3_primary_rule_sensors)),
    "stage3_secondary_rule_count": int(len(stage3_secondary_rule_sensors)),
    "stage2_selection_mode": STAGE2_SELECTION_MODE,
    "stage2_best_params": stage2_best_params,
}

truth_config_object = globals().get("TRUTH_CONFIG")
run_mode_value = globals().get("RUN_MODE")
config_profile_value = globals().get("CONFIG_PROFILE", "default")
gold_process_run_id_value = globals().get("GOLD_PROCESS_RUN_ID")

if isinstance(truth_config_object, dict):
    truth_config_snapshot = truth_config_object
else:
    truth_config_snapshot = {
        "runtime": {
            "stage": "gold_cascade",
            "dataset": DATASET_NAME,
            "cascade_variant": CASCADE_VARIANT,
            "mode": run_mode_value,
            "profile": config_profile_value,
        }
    }

cascade_truth_layer_name = "gold_cascade"

if isinstance(gold_process_run_id_value, str) and gold_process_run_id_value.strip():
    cascade_process_run_id = gold_process_run_id_value
else:
    cascade_process_run_id = make_process_run_id("gold_cascade_process")

cascade_truth = initialize_layer_truth(
    truth_version=TRUTH_VERSION,
    dataset_name=DATASET_NAME,
    layer_name=cascade_truth_layer_name,
    process_run_id=cascade_process_run_id,
    pipeline_mode=PIPELINE_MODE,
    parent_truth_hash=GOLD_PARENT_TRUTH_HASH,
)

cascade_truth = update_truth_section(
    cascade_truth,
    "config_snapshot",
    truth_config_snapshot,
)

cascade_truth = update_truth_section(
    cascade_truth,
    "runtime_facts",
    {
        "cascade_variant": CASCADE_VARIANT,
        "stage1_threshold_percentile": float(STAGE1_THRESHOLD_PERCENTILE),
        "stage1_threshold": float(stage1_threshold),
        "stage2_selection_mode": STAGE2_SELECTION_MODE,
        "stage2_selected_threshold_percentile": float(stage2_selected_threshold_percentile),
        "stage2_threshold": float(stage2_threshold),
        "stage1_estimator_count": int(STAGE1_ESTIMATOR_COUNT),
        "stage2_estimator_count": int(stage2_model.get_params()["n_estimators"]),
        "stage2_best_params": stage2_best_params,
        "stage1_feature_count": int(len(stage1_feature_columns)),
        "stage2_feature_count": int(len(stage2_feature_columns)),
        "stage3_primary_rule_count": int(len(stage3_primary_rule_sensors)),
        "stage3_secondary_rule_count": int(len(stage3_secondary_rule_sensors)),
        "result_row_count": int(len(cascade_results)),
        "parent_truth_hash": GOLD_PARENT_TRUTH_HASH,
        "gold_process_run_id": gold_truth.get("process_run_id"),
        "gold_feature_set_id": gold_truth_runtime_facts.get("feature_set_id"),
    },
)

cascade_truth = update_truth_section(
    cascade_truth,
    "artifact_paths",
    {
        "cascade_variant": CASCADE_VARIANT,
        "gold_truth_path": str(GOLD_TRUTH_PATH),
        "gold_fit_path": str(GOLD_FIT_DATA_PATH),
        "gold_scored_path": str(GOLD_PREPROCESSED_SCALED_DATA_PATH),
        "stage1_features_path": str(STAGE1_FEATURES_PATH),
        "stage2_features_path": str(STAGE2_FEATURES_PATH),
        "stage3_primary_path": str(STAGE3_PRIMARY_PATH),
        "stage3_secondary_path": str(STAGE3_SECONDARY_PATH),
        "cascade_defaults_results_path_csv": str(CASCADE_RESULTS_PATH_CSV),
        "cascade_defaults_results_path_pickle": str(CASCADE_RESULTS_PATH_PICKLE),
        "cascade_defaults_stage1_model_artifact_path": str(STAGE1_MODEL_ARTIFACT_PATH),
        "cascade_defaults_stage1_models_path": str(STAGE1_MODELS_PATH),
        "cascade_defaults_stage2_model_artifact_path": str(STAGE2_MODEL_ARTIFACT_PATH),
        "cascade_defaults_stage2_models_path": str(STAGE2_MODELS_PATH),
        "cascade_defaults_thresholds_path": str(CASCADE_THRESHOLDS_PATH),
        "cascade_defaults_summary_path": str(CASCADE_SUMMARY_PATH),
        "cascade_defaults_metadata_path": str(CASCADE_METADATA_PATH),
        "cascade_defaults_reference_profile_path": str(CASCADE_REFERENCE_PROFILE_PATH),
    },
)

cascade_meta_columns = sorted(
    set(
        identify_meta_columns(cascade_results)
        + [
            "meta__truth_hash",
            "meta__parent_truth_hash",
            "meta__pipeline_mode",
        ]
    )
)

cascade_feature_columns = identify_feature_columns(cascade_results)

cascade_truth_record = build_truth_record(
    truth_base=cascade_truth,
    row_count=len(cascade_results),
    column_count=cascade_results.shape[1] + 3,
    meta_columns=cascade_meta_columns,
    feature_columns=cascade_feature_columns,
)

CASCADE_TRUTH_HASH = cascade_truth_record["truth_hash"]

cascade_results = stamp_truth_columns(
    cascade_results,
    truth_hash=CASCADE_TRUTH_HASH,
    parent_truth_hash=GOLD_PARENT_TRUTH_HASH,
    pipeline_mode=PIPELINE_MODE,
)

cascade_truth_path = save_truth_record(
    cascade_truth_record,
    truth_dir=TRUTHS_PATH,
    dataset_name=DATASET_NAME,
    layer_name=cascade_truth_layer_name,
)

append_truth_index(
    cascade_truth_record,
    truth_index_path=TRUTH_INDEX_PATH,
)

cascade_summary["cascade_truth_hash"] = CASCADE_TRUTH_HASH
cascade_summary["cascade_truth_path"] = str(cascade_truth_path)
cascade_summary["cascade_process_run_id"] = cascade_process_run_id
cascade_summary["gold_truth_hash"] = GOLD_PARENT_TRUTH_HASH
cascade_summary["gold_truth_path"] = str(GOLD_TRUTH_PATH)
cascade_summary["gold_process_run_id"] = gold_truth.get("process_run_id")
cascade_summary["gold_feature_set_id"] = gold_truth_runtime_facts.get("feature_set_id")

display(cascade_summary)

cascade_metadata = {
    "cascade_variant": CASCADE_VARIANT,
    "cascade_defaults_results_path_csv": str(CASCADE_RESULTS_PATH_CSV),
    "cascade_defaults_results_path_pickle": str(CASCADE_RESULTS_PATH_PICKLE),
    "cascade_defaults_stage2_selection_mode": STAGE2_SELECTION_MODE,
    "cascade_defaults_stage1_model_artifact_path": str(STAGE1_MODEL_ARTIFACT_PATH),
    "cascade_defaults_stage1_models_path": str(STAGE1_MODELS_PATH),
    "cascade_defaults_stage2_model_artifact_path": str(STAGE2_MODEL_ARTIFACT_PATH),
    "cascade_defaults_stage2_models_path": str(STAGE2_MODELS_PATH),
    "cascade_defaults_thresholds_path": str(CASCADE_THRESHOLDS_PATH),
    "cascade_defaults_summary_path": str(CASCADE_SUMMARY_PATH),
    "cascade_defaults_metadata_path": str(CASCADE_METADATA_PATH),
    "cascade_defaults_reference_profile_path": str(CASCADE_REFERENCE_PROFILE_PATH),
    "gold_fit_path": str(GOLD_FIT_DATA_PATH),
    "gold_scored_path": str(GOLD_PREPROCESSED_SCALED_DATA_PATH),

    # upstream truth linkage
    "gold_truth_hash": GOLD_PARENT_TRUTH_HASH,
    "gold_truth_path": str(GOLD_TRUTH_PATH),
    "gold_process_run_id": gold_truth.get("process_run_id"),
    "gold_feature_set_id": gold_truth_runtime_facts.get("feature_set_id"),
    "gold_scaler_path": gold_truth_artifact_paths.get("scaler_path"),
    "gold_scaler_kind": gold_truth_runtime_facts.get("scaler_kind_runtime"),
    "gold_recommended_imputation": gold_truth_runtime_facts.get("recommended_imputation"),

    # stage truth linkage
    "cascade_truth_hash": CASCADE_TRUTH_HASH,
    "cascade_truth_path": str(cascade_truth_path),
    "cascade_process_run_id": cascade_process_run_id,
}
display(cascade_metadata)

display(cascade_results)
cascade_results.to_csv(CASCADE_RESULTS_PATH_CSV, index=False)
cascade_results.to_pickle(CASCADE_RESULTS_PATH_PICKLE)

display(reference_profile)
reference_profile.to_csv(CASCADE_REFERENCE_PROFILE_PATH, index=False)


joblib.dump(stage1_model, STAGE1_MODEL_ARTIFACT_PATH)
joblib.dump(stage1_model, STAGE1_MODELS_PATH)

joblib.dump(stage2_model, STAGE2_MODEL_ARTIFACT_PATH)
joblib.dump(stage2_model, STAGE2_MODELS_PATH)

save_json(cascade_thresholds, CASCADE_THRESHOLDS_PATH)
save_json(cascade_summary, CASCADE_SUMMARY_PATH)
save_json(cascade_metadata, CASCADE_METADATA_PATH)

wandb.save(str(CASCADE_RESULTS_PATH_CSV))
wandb.save(str(CASCADE_RESULTS_PATH_PICKLE))
wandb.save(str(CASCADE_REFERENCE_PROFILE_PATH))
wandb.save(str(STAGE1_MODEL_ARTIFACT_PATH))
wandb.save(str(STAGE1_MODELS_PATH))
wandb.save(str(STAGE2_MODEL_ARTIFACT_PATH))
wandb.save(str(STAGE2_MODELS_PATH))
wandb.save(str(CASCADE_THRESHOLDS_PATH))
wandb.save(str(CASCADE_SUMMARY_PATH))
wandb.save(str(CASCADE_METADATA_PATH))
wandb.save(str(cascade_truth_path))

ledger.add(
    kind="step",
    step="save_cascade_outputs",
    message="Saved cascade results, trained Stage 1 and Stage 2 models, thresholds, summary, metadata, reference profile, and cascade stage truth record.",
    data={
        "cascade_variant": CASCADE_VARIANT,
        "stage2_selection_mode": STAGE2_SELECTION_MODE,
        "cascade_defaults_results_path_csv": str(CASCADE_RESULTS_PATH_CSV),
        "cascade_defaults_results_path_pickle": str(CASCADE_RESULTS_PATH_PICKLE),
        "cascade_defaults_reference_profile_path": str(CASCADE_REFERENCE_PROFILE_PATH),
        "cascade_defaults_stage1_model_artifact_path": str(STAGE1_MODEL_ARTIFACT_PATH),
        "cascade_defaults_stage1_models_path": str(STAGE1_MODELS_PATH),
        "cascade_defaults_stage2_model_artifact_path": str(STAGE2_MODEL_ARTIFACT_PATH),
        "cascade_defaults_stage2_models_path": str(STAGE2_MODELS_PATH),
        "cascade_defaults_thresholds_path": str(CASCADE_THRESHOLDS_PATH),
        "cascade_defaults_summary_path": str(CASCADE_SUMMARY_PATH),
        "cascade_defaults_metadata_path": str(CASCADE_METADATA_PATH),
        "cascade_truth_hash": CASCADE_TRUTH_HASH,
        "cascade_truth_path": str(cascade_truth_path),
        "result_row_count": int(len(cascade_results)),
        "final_alert_count_all_rows": final_cascade_alert_count_all_rows,
        "final_alert_count_test_rows": final_cascade_alert_count_test_rows,
    },
    logger=logger,
)

{'dataset_name': 'pump',
 'cascade_variant': 'default',
 'cascade_metrics': {'model': '3-Stage Cascade',
  'stage1_alert_count_all_rows': 103054,
  'stage2_alert_count_all_rows': 70376,
  'final_alert_count_all_rows': 69362,
  'stage1_alert_count_test_rows': 43236,
  'stage2_alert_count_test_rows': 25738,
  'final_alert_count_test_rows': 24895,
  'precision': 0.0030929905603534844,
  'recall': 0.652542372881356,
  'f1': 0.006156798464798305},
 'stage1_alert_count_all_rows': 103054,
 'stage2_alert_count_all_rows': 70376,
 'final_cascade_alert_count_all_rows': 69362,
 'stage1_alert_count_test_rows': 43236,
 'stage2_alert_count_test_rows': 25738,
 'final_cascade_alert_count_test_rows': 24895,
 'result_row_count': 220320,
 'stage1_feature_count': 50,
 'stage2_feature_count': 12,
 'stage3_primary_rule_count': 5,
 'stage3_secondary_rule_count': 8,
 'stage2_selection_mode': 'fixed',
 'stage2_best_params': {'n_estimators': 300,
  'max_samples': 'auto',
  'contamination': 'auto',
  'max_feature

{'cascade_variant': 'default',
 'cascade_defaults_results_path_csv': '/workspace/artifacts/gold/pump/cascade_defaults/scores/pump__gold__cascade_defaults_results.csv',
 'cascade_defaults_results_path_pickle': '/workspace/artifacts/gold/pump/cascade_defaults/scores/pump__gold__cascade_defaults_results.pkl',
 'cascade_defaults_stage2_selection_mode': 'fixed',
 'cascade_defaults_stage1_model_artifact_path': '/workspace/artifacts/gold/pump/cascade_defaults/models/pump__gold__cascade_defaults_stage1_isolation_forest.joblib',
 'cascade_defaults_stage1_models_path': '/workspace/models/pump/pump__gold__cascade_defaults_stage1_isolation_forest.joblib',
 'cascade_defaults_stage2_model_artifact_path': '/workspace/artifacts/gold/pump/cascade_defaults/models/pump__gold__cascade_defaults_stage2_isolation_forest.joblib',
 'cascade_defaults_stage2_models_path': '/workspace/models/pump/pump__gold__cascade_defaults_stage2_isolation_forest.joblib',
 'cascade_defaults_thresholds_path': '/workspace/artifac

,meta__asset_id,meta__dataset,meta__episode_id,meta__event_id,meta__ingested_at_utc,meta__parent_truth_hash,meta__pipeline_mode,meta__record_id,meta__run_id,meta__source_file,meta__source_row_id,meta__split,meta__truth_hash,event_time,event_step,time_index,event_date,anomaly_flag,is_anomaly,is_normal,status_normal_value,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,...,sensor_47__delta_abnormal_flag,sensor_47__any_abnormal_flag,sensor_48__value_deviation,sensor_48__delta_deviation,sensor_48__value_abnormal_flag,sensor_48__delta_abnormal_flag,sensor_48__any_abnormal_flag,sensor_49__value_deviation,sensor_49__delta_deviation,sensor_49__value_abnormal_flag,sensor_49__delta_abnormal_flag,sensor_49__any_abnormal_flag,sensor_51__value_deviation,sensor_51__delta_deviation,sensor_51__value_abnormal_flag,sensor_51__delta_abnormal_flag,sensor_51__any_abnormal_flag,normal_value_abnormal_sensor_count,normal_delta_abnormal_sensor_count,normal_total_abnormal_sensor_count,normal_training_quality_class,is_clean_normal_for_training,final_row_quality_class,row_is_clean_normal,row_is_suspect_normal,row_is_exclude_from_normal_training,machine_status__profiled,meta__row_id,meta__is_train_flag,stage1_score,stage1_decision,stage1_pred,stage1_flag,stage1_threshold,stage1_threshold_percentile,stage2_model_score,stage2_model_decision,stage2_model_pred,stage2_model_flag,stage2_score,stage2_raw_flag,stage2_flag,stage3_profile_breach_count,stage3_secondary_breach_count,stage3_persistence_flag,stage3_drift_flag,stage3_profile_breach_flag,stage3_corroboration_flag,stage3_rule_evidence_count,cascade_final_flag
0,asset__001,pump,0,pump:asset__001:run__001:0,2026-06-14 20:37:14.939771+00:00,1b39755619d21614721b74a063b90aceddebaa1f4b8573...,batch,14598431322315673869,run__001,sensor.csv,0,train,70cc8fe26c311e8ff807cdd4c5151cafcf2637f37d197d...,2018-04-01 00:00:00+00:00,0,0,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,0.140011,-0.695652,0.464286,1.269226,0.108436,-0.168590,-0.369557,0.124978,0.736837,-0.266728,-1.260975,0.925032,-0.034035,-0.123905,-0.461029,-0.672626,0.304680,0.142472,-0.295277,-0.461054,-0.091023,-0.172259,-0.121836,0.099551,-0.311720,-0.479814,-0.666538,-1.737734,-0.546495,...,False,False,0.716570,NaN,False,False,False,2.780487,NaN,False,False,False,0.275862,NaN,False,False,False,4,0,4,clean,True,clean,True,False,False,normal_clean,0,True,0.409368,0.090632,1,0,0.488648,95.0,NaN,NaN,NaN,NaN,NaN,0,0,1,2,0,0,0,1,1,0
1,asset__001,pump,0,pump:asset__001:run__001:1,2026-06-14 20:37:14.939771+00:00,1b39755619d21614721b74a063b90aceddebaa1f4b8573...,batch,15954729095895098000,run__001,sensor.csv,1,train,70cc8fe26c311e8ff807cdd4c5151cafcf2637f37d197d...,2018-04-01 00:01:00+00:00,1,1,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,0.140011,-0.695652,0.464286,1.269226,0.108436,-0.168590,-0.369557,0.124978,0.736837,-0.266728,-1.260975,0.925032,-0.034035,-0.123905,-0.461029,-0.672626,0.304680,0.142472,-0.295277,-0.461054,-0.091023,-0.172259,-0.121836,0.099551,-0.311720,-0.479814,-0.666538,-1.737734,-0.546495,...,False,False,0.716570,0.000000,False,False,False,2.780487,0.000000,False,False,False,0.275862,0.000000,False,False,False,4,0,4,clean,True,clean,True,False,False,normal_clean,1,True,0.409368,0.090632,1,0,0.488648,95.0,NaN,NaN,NaN,NaN,NaN,0,0,1,2,0,0,0,1,1,0
2,asset__001,pump,0,pump:asset__001:run__001:2,2026-06-14 20:37:14.939771+00:00,1b39755619d21614721b74a063b90aceddebaa1f4b8573...,batch,10041703297090838359,run__001,sensor.csv,2,train,70cc8fe26c311e8ff807cdd4c5151cafcf2637f37d197d...,2018-04-01 00:02:00+00:00,2,2,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,-0.280002,-0.565216,0.464286,1.307690,0.578311,-0.537436,-0.630443,-0.078128,0.859651,-0.666728,-1.160344,1.006862,0.146473,-0.108279,0.230324,-0.259305,-0.439206,-0.506023,

,feature_name,median_value,mean_value,standard_deviation,lower_bound,upper_bound
0,sensor_00,0.0,-0.225564,1.983531,-2.540039,1.100022
1,sensor_01,0.0,0.021708,0.841543,-1.347824,1.456523
2,sensor_02,0.0,-0.011922,0.672431,-1.089288,1.017860
3,sensor_03,0.0,0.060602,0.691585,-0.980768,1.249997
4,sensor_04,0.0,-0.232387,1.951311,-1.927685,1.228905
5,sensor_05,0.0,0.015411,0.868819,-1.361052,1.536601
6,sensor_06,0.0,-0.038234,1.160401,-1.434801,1.804388
7,sensor_07,0.0,-0.164475,0.692269,-1.046894,1.156255
8,sensor_08,0.0,0.147561,0.763275,-0.929813,1.333325
9,sensor_09,0.0,-0.556567,3.136236,-2.666912,0.733364


2026-06-16 18:14:56,913 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/cascade_defaults/thresholds/pump__gold__cascade_defaults_thresholds.json
2026-06-16 18:14:57,014 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/cascade_defaults/summaries/pump__gold__cascade_defaults_summary.json
2026-06-16 18:14:57,108 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/cascade_defaults/metadata/pump__gold__cascade_defaults_metadata.json
wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
2026-06-16 18:14:58,333 | INFO | capstone.gold.cascade.defaults | LEDGER | {'ts_utc': '2026-06-16T18:14:58.333044+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'save_cascade_outputs', 'message': 'Saved cascade results, trained Stage 1 and Stage 2 models, thresholds, summary, metad

{'ts_utc': '2026-06-16T18:14:58.333044+00:00',
 'stage': 'gold_cascade',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'save_cascade_outputs',
 'message': 'Saved cascade results, trained Stage 1 and Stage 2 models, thresholds, summary, metadata, reference profile, and cascade stage truth record.',
 'why': None,
 'consequence': None,
 'data': {'cascade_variant': 'default',
  'stage2_selection_mode': 'fixed',
  'cascade_defaults_results_path_csv': '/workspace/artifacts/gold/pump/cascade_defaults/scores/pump__gold__cascade_defaults_results.csv',
  'cascade_defaults_results_path_pickle': '/workspace/artifacts/gold/pump/cascade_defaults/scores/pump__gold__cascade_defaults_results.pkl',
  'cascade_defaults_reference_profile_path': '/workspace/artifacts/gold/pump/cascade_defaults/profiles/pump__gold__cascade_defaults_reference_profile.csv',
  'cascade_defaults_stage1_model_artifact_path': '/workspace/artifacts/gold/pump/cascade_defaults/models/pump__gold__cascade_default

----

In [48]:
stage2_percentile_for_contract = float(stage2_selected_threshold_percentile)

cascade_default_contract_path = gold_model_validation_contract_path(
    artifacts_root=paths.artifacts,
    dataset_id=DATASET_ID,
    model_id="cascade_default",
)

cascade_default_contract = build_gold_model_output_validation_contract(
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    model_id="cascade_default",
    model_label="Cascade Default",
    source_notebook="gold_03a_cascade_modeling",
    validation_type="cascade_rule_artifact",
    model_stage="cascade_default_final",
    operating_mode="default",
    metrics=cascade_metrics,
    output_dataframe=cascade_results,
    output_flag_column="cascade_final_flag",
    test_mask=test_mask,
    rule_config={
        "cascade_variant": CASCADE_VARIANT,
        "stage1_threshold_percentile": float(STAGE1_THRESHOLD_PERCENTILE),
        "stage1_threshold": float(stage1_threshold),
        "stage2_selection_mode": STAGE2_SELECTION_MODE,
        "stage2_selected_threshold_percentile": stage2_percentile_for_contract,
        "stage2_threshold": float(stage2_threshold),
        "stage2_best_params": stage2_best_params,
        "stage3_min_primary_sensor_hits": int(STAGE3_MIN_PRIMARY_SENSOR_HITS),
        "stage3_min_secondary_sensor_hits": int(STAGE3_MIN_SECONDARY_SENSOR_HITS),
        "stage3_rolling_window_size": int(STAGE3_ROLLING_WINDOW_SIZE),
        "stage3_minimum_flags_in_window": int(STAGE3_MINIMUM_FLAGS_IN_WINDOW),
    },
    rule_source="gold_03a_summary_thresholds_and_stage3_rule_cells",
    stage3_type="rule_based",
    stage3_saved_as_joblib=False,
    stage1_model_path=STAGE1_MODEL_ARTIFACT_PATH,
    stage2_model_path=STAGE2_MODEL_ARTIFACT_PATH,
    output_artifact_path=CASCADE_RESULTS_PATH_CSV,
    lineage_payload={
        "cascade_truth_hash": globals().get("CASCADE_TRUTH_HASH"),
        "parent_gold_truth_hash": globals().get("GOLD_PARENT_TRUTH_HASH"),
    },
    notes="Gold 03a default cascade contract. Stage 3 is rule-based and not saved as joblib.",
)

write_gold_model_output_validation_contract(
    contract=cascade_default_contract,
    output_path=cascade_default_contract_path,
)

ledger.add(
    kind="artifact",
    step="gold03a_cascade_default_validation_contract_written",
    message="Wrote Gold 03a cascade-default validation contract for Gold 06.",
    data={
        "path": str(cascade_default_contract_path),
        "model_id": cascade_default_contract["model_id"],
        "alert_count_test_rows": cascade_default_contract["metric_payload"]["alert_count_test_rows"],
    },
    logger=logger,
)

display(cascade_default_contract_path)

2026-06-16 18:14:59,593 | INFO | capstone.gold.cascade.defaults | LEDGER | {'ts_utc': '2026-06-16T18:14:59.593360+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'artifact', 'step': 'gold03a_cascade_default_validation_contract_written', 'message': 'Wrote Gold 03a cascade-default validation contract for Gold 06.', 'why': None, 'consequence': None, 'data': {'path': '/workspace/artifacts/gold/pump/model_validation/contracts/pump__gold__cascade_default_validation_contract.json', 'model_id': 'cascade_default', 'alert_count_test_rows': 24895}}


PosixPath('/workspace/artifacts/gold/pump/model_validation/contracts/pump__gold__cascade_default_validation_contract.json')

----

In [49]:
stage1_detected_rows_dataframe = get_detected_rows_dataframe(
    dataframe=cascade_results,
    target_flag_column="stage1_flag",
    row_id_column="meta__row_id",
    score_column="stage1_score",
    decision_column="stage1_decision",
    pred_column="stage1_pred",
    include_columns=[
        "stage1_flag",
        "stage2_raw_flag",
        "stage2_flag",
        "cascade_final_flag",
        "anomaly_flag",
    ],
    sort_by="time_index",
    ascending=True,
)

ledger.add(
    kind="step",
    step="extract_stage1_detected_rows",
    message="Built the Stage 1 detected-rows dataframe from the cascade results using stable row tracking.",
    data={
        "target_flag_column": "stage1_flag",
        "row_id_column": "meta__row_id",
        "detected_row_count": int(len(stage1_detected_rows_dataframe)),
        "has_time_index": bool("time_index" in stage1_detected_rows_dataframe.columns),
        "has_event_step": bool("event_step" in stage1_detected_rows_dataframe.columns),
        "has_event_time": bool("event_time" in stage1_detected_rows_dataframe.columns),
    },
    logger=logger,
)

print(f"Stage 1 detected row count: {len(stage1_detected_rows_dataframe):,}")
display(stage1_detected_rows_dataframe.head(20))

2026-06-16 18:15:00,959 | INFO | capstone.gold.cascade.defaults | LEDGER | {'ts_utc': '2026-06-16T18:15:00.959029+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'extract_stage1_detected_rows', 'message': 'Built the Stage 1 detected-rows dataframe from the cascade results using stable row tracking.', 'why': None, 'consequence': None, 'data': {'target_flag_column': 'stage1_flag', 'row_id_column': 'meta__row_id', 'detected_row_count': 103054, 'has_time_index': True, 'has_event_step': True, 'has_event_time': True}}


Stage 1 detected row count: 103,054


,meta__row_id,meta__record_id,time_index,event_step,event_time,meta__asset_id,meta__run_id,machine_status,anomaly_flag,stage1_flag,stage1_score,stage1_decision,stage1_pred,stage2_raw_flag,stage2_flag,cascade_final_flag
0,260,13356824106279790836,260,260,2018-04-01 04:20:00+00:00,asset__001,run__001,NORMAL,0,1,0.493609,0.006391,1,0,0,0
1,262,14875396535299734039,262,262,2018-04-01 04:22:00+00:00,asset__001,run__001,NORMAL,0,1,0.493296,0.006704,1,0,0,0
2,265,9319389479859232848,265,265,2018-04-01 04:25:00+00:00,asset__001,run__001,NORMAL,0,1,0.500847,-0.000847,-1,0,0,0
3,266,7893936890144193503,266,266,2018-04-01 04:26:00+00:00,asset__001,run__001,NORMAL,0,1,0.496788,0.003212,1,0,0,0
4,267,7750815740698537469,267,267,2018-04-01 04:27:00+00:00,asset__001,run__001,NORMAL,0,1,0.496377,0.003623,1,0,0,0
5,268,17198309132295013087,268,268,2018-04-01 04:28:00+00:00,asset__001,run__001,NORMAL,0,1,0.498531,0.001469,1,0,0,0
6,269,13285358529408335145,269,269,2018-04-01 04:29:00+00:00,asset__001,run__001,NORMAL,0,1,0.494670,0.005330,1,0,0,0
7,270,8119318478283180666,270,270,2018-04-01 04:30:00+00:00,asset__001,run__001,NORMAL,0,1,0.500057,-0.000057,-1,0,0,0
8,271,2333674238325070709,271,271,2018-04-01 04:31:00+00:00,asset__001,run__001,NORMAL,0,1,0.500683,-0.000683,-1,0,0,0
9,272,18100572650003074832,272,272,2018-04-01 04:32:00+00:00,asset__001,run__001,NORMAL,0,1,0.496375,0.003625,1,0,0,0


#### Build detected-row review extract

This cell records traceability information for the current transformation. The ledger/log output documents what changed, why it changed, and how the step should be audited.

In [50]:
stage2_detected_rows_dataframe = get_detected_rows_dataframe(
    dataframe=cascade_results,
    target_flag_column="stage2_flag",
    row_id_column="meta__row_id",
    score_column="stage2_score",
    decision_column="stage2_model_decision",
    pred_column="stage2_model_pred",
    include_columns=[
        "stage1_flag",
        "stage2_raw_flag",
        "stage2_flag",
        "stage2_model_score",
        "stage2_model_decision",
        "stage2_model_pred",
        "stage2_model_flag",
        "cascade_final_flag",
        "anomaly_flag",
    ],
    sort_by="time_index",
    ascending=True,
)

ledger.add(
    kind="step",
    step="extract_stage2_detected_rows",
    message="Built the Stage 2 detected-rows dataframe from the cascade results using stable row tracking.",
    data={
        "target_flag_column": "stage2_flag",
        "row_id_column": "meta__row_id",
        "detected_row_count": int(len(stage2_detected_rows_dataframe)),
        "has_time_index": bool("time_index" in stage2_detected_rows_dataframe.columns),
        "has_event_step": bool("event_step" in stage2_detected_rows_dataframe.columns),
        "has_event_time": bool("event_time" in stage2_detected_rows_dataframe.columns),
    },
    logger=logger,
)

print(f"Stage 2 detected row count: {len(stage2_detected_rows_dataframe):,}")
display(stage2_detected_rows_dataframe.head(20))

2026-06-16 18:15:01,885 | INFO | capstone.gold.cascade.defaults | LEDGER | {'ts_utc': '2026-06-16T18:15:01.885135+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'extract_stage2_detected_rows', 'message': 'Built the Stage 2 detected-rows dataframe from the cascade results using stable row tracking.', 'why': None, 'consequence': None, 'data': {'target_flag_column': 'stage2_flag', 'row_id_column': 'meta__row_id', 'detected_row_count': 70376, 'has_time_index': True, 'has_event_step': True, 'has_event_time': True}}


Stage 2 detected row count: 70,376


,meta__row_id,meta__record_id,time_index,event_step,event_time,meta__asset_id,meta__run_id,machine_status,anomaly_flag,stage2_flag,stage2_score,stage2_model_decision,stage2_model_pred,stage1_flag,stage2_raw_flag,stage2_model_score,stage2_model_flag,cascade_final_flag
0,5202,12842405811221279513,5202,5202,2018-04-04 14:42:00+00:00,asset__001,run__001,NORMAL,0,1,0.568476,-0.068476,-1.0,1,1,-0.568476,1.0,1
1,5205,11598710423751654092,5205,5205,2018-04-04 14:45:00+00:00,asset__001,run__001,NORMAL,0,1,0.572935,-0.072935,-1.0,1,1,-0.572935,1.0,1
2,8195,1628843015739861844,8195,8195,2018-04-06 16:35:00+00:00,asset__001,run__001,NORMAL,0,1,0.567308,-0.067308,-1.0,1,1,-0.567308,1.0,1
3,8239,14617586387100313698,8239,8239,2018-04-06 17:19:00+00:00,asset__001,run__001,NORMAL,0,1,0.571852,-0.071852,-1.0,1,1,-0.571852,1.0,1
4,8241,12991760080095676024,8241,8241,2018-04-06 17:21:00+00:00,asset__001,run__001,NORMAL,0,1,0.566312,-0.066312,-1.0,1,1,-0.566312,1.0,1
5,8246,11408035136458572257,8246,8246,2018-04-06 17:26:00+00:00,asset__001,run__001,NORMAL,0,1,0.574100,-0.074100,-1.0,1,1,-0.574100,1.0,0
6,8248,15244127189313681115,8248,8248,2018-04-06 17:28:00+00:00,asset__001,run__001,NORMAL,0,1,0.567661,-0.067661,-1.0,1,1,-0.567661,1.0,1
7,8249,3622191867065128171,8249,8249,2018-04-06 17:29:00+00:00,asset__001,run__001,NORMAL,0,1,0.581498,-0.081498,-1.0,1,1,-0.581498,1.0,1
8,10614,6522429923118725157,10614,10614,2018-04-08 08:54:00+00:00,asset__001,run__001,NORMAL,0,1,0.566434,-0.066434,-1.0,1,1,-0.566434,1.0,1
9,10615,5800922206161343483,10615,10615,2018-04-08 08:55:00+00:00,asset__001,run__001,NORMAL,0,1,0.572049,-0.072049,-1.0,1,1,-0.572049,1.0,1


#### Build detected-row review extract

This cell records traceability information for the current transformation. The ledger/log output documents what changed, why it changed, and how the step should be audited.

In [ ]:
stage3_evidence_rows_dataframe = get_detected_rows_dataframe(
    dataframe=cascade_results,
    target_flag_column="stage3_profile_breach_flag",
    row_id_column="meta__row_id",
    include_columns=[
        "stage1_flag",
        "stage2_raw_flag",
        "stage2_flag",
        "stage3_profile_breach_count",
        "stage3_secondary_breach_count",
        "stage3_profile_breach_flag",
        "stage3_corroboration_flag",
        "stage3_persistence_flag",
        "stage3_drift_flag",
        "stage3_rule_evidence_count",
        "cascade_final_flag",
        "anomaly_flag",
    ],
    sort_by="time_index",
    ascending=True,
)

ledger.add(
    kind="step",
    step="extract_stage3_evidence_rows",
    message="Built the Stage 3 evidence-row dataframe from the cascade results using stable row tracking.",
    data={
        "target_flag_column": "stage3_profile_breach_flag",
        "row_id_column": "meta__row_id",
        "detected_row_count": int(len(stage3_evidence_rows_dataframe)),
    },
    logger=logger,
)

print(f"Stage 3 evidence row count: {len(stage3_evidence_rows_dataframe):,}")
display(stage3_evidence_rows_dataframe.head(20))

2026-06-16 18:15:03,168 | INFO | capstone.gold.cascade.defaults | LEDGER | {'ts_utc': '2026-06-16T18:15:03.168413+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'extract_stage3_evidence_rows', 'message': 'Built the Stage 3 evidence-row dataframe from the cascade results using stable row tracking.', 'why': None, 'consequence': None, 'data': {'target_flag_column': 'stage3_profile_breach_flag', 'row_id_column': 'meta__row_id', 'detected_row_count': 92627}}


Stage 3 evidence row count: 92,627


,meta__row_id,meta__record_id,time_index,event_step,event_time,meta__asset_id,meta__run_id,machine_status,anomaly_flag,stage3_profile_breach_flag,stage1_flag,stage2_raw_flag,stage2_flag,stage3_profile_breach_count,stage3_secondary_breach_count,stage3_corroboration_flag,stage3_persistence_flag,stage3_drift_flag,stage3_rule_evidence_count,cascade_final_flag
0,3,2072036352569063261,3,3,2018-04-01 00:03:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,1,1,0,0,2,0
1,4,4365040424004714369,4,4,2018-04-01 00:04:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,1,1,0,0,2,0
2,6,7206199315957152377,6,6,2018-04-01 00:06:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,1,1,0,0,2,0
3,8,4003627476009805685,8,8,2018-04-01 00:08:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,1,1,0,0,2,0
4,9,12843418223340544546,9,9,2018-04-01 00:09:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,1,1,0,0,2,0
5,10,12764708609319381380,10,10,2018-04-01 00:10:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,3,1,1,0,0,2,0
6,11,17692035958682023704,11,11,2018-04-01 00:11:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,2,1,0,0,2,0
7,12,9921446699215930665,12,12,2018-04-01 00:12:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,1,1,0,0,2,0
8,13,9345101619896030940,13,13,2018-04-01 00:13:00+00:00,asset__001,run__001,NORMAL,0,1,0,0,0,2,2,1,0,0,2,0
9,363,9387970151524258740,363,363,2018-04-01 06:03:00+00:00,asset__001,run__001,NORMAL,0,1,1,0,0,2,1,1,0,0,2,0


#### Build detected-row review extract

This cell records traceability information for the current transformation. The ledger/log output documents what changed, why it changed, and how the step should be audited.

In [52]:
final_detected_rows_dataframe = get_detected_rows_dataframe(
    dataframe=cascade_results,
    target_flag_column="cascade_final_flag",
    row_id_column="meta__row_id",
    score_column="stage2_score",
    decision_column="stage2_model_decision",
    pred_column="stage2_model_pred",
    include_columns=[
        "stage1_flag",
        "stage2_raw_flag",
        "stage2_flag",
        "stage2_model_score",
        "stage2_model_decision",
        "stage2_model_pred",
        "stage2_model_flag",
        "stage3_profile_breach_count",
        "stage3_secondary_breach_count",
        "stage3_profile_breach_flag",
        "stage3_corroboration_flag",
        "stage3_persistence_flag",
        "stage3_drift_flag",
        "stage3_rule_evidence_count",
        "cascade_final_flag",
        "anomaly_flag",
    ],
    sort_by="time_index",
    ascending=True,
)

ledger.add(
    kind="step",
    step="extract_final_detected_rows",
    message="Built the final detected-rows dataframe from the cascade results using stable row tracking.",
    data={
        "target_flag_column": "cascade_final_flag",
        "row_id_column": "meta__row_id",
        "detected_row_count": int(len(final_detected_rows_dataframe)),
        "has_time_index": bool("time_index" in final_detected_rows_dataframe.columns),
        "has_event_step": bool("event_step" in final_detected_rows_dataframe.columns),
        "has_event_time": bool("event_time" in final_detected_rows_dataframe.columns),
    },
    logger=logger,
)

print(f"Final cascade detected row count: {len(final_detected_rows_dataframe):,}")
display(final_detected_rows_dataframe.head(20))

2026-06-16 18:15:04,053 | INFO | capstone.gold.cascade.defaults | LEDGER | {'ts_utc': '2026-06-16T18:15:04.053908+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'extract_final_detected_rows', 'message': 'Built the final detected-rows dataframe from the cascade results using stable row tracking.', 'why': None, 'consequence': None, 'data': {'target_flag_column': 'cascade_final_flag', 'row_id_column': 'meta__row_id', 'detected_row_count': 69362, 'has_time_index': True, 'has_event_step': True, 'has_event_time': True}}


Final cascade detected row count: 69,362


,meta__row_id,meta__record_id,time_index,event_step,event_time,meta__asset_id,meta__run_id,machine_status,anomaly_flag,cascade_final_flag,stage2_score,stage2_model_decision,stage2_model_pred,stage1_flag,stage2_raw_flag,stage2_flag,stage2_model_score,stage2_model_flag,stage3_profile_breach_count,stage3_secondary_breach_count,stage3_profile_breach_flag,stage3_corroboration_flag,stage3_persistence_flag,stage3_drift_flag,stage3_rule_evidence_count
0,5202,12842405811221279513,5202,5202,2018-04-04 14:42:00+00:00,asset__001,run__001,NORMAL,0,1,0.568476,-0.068476,-1.0,1,1,1,-0.568476,1.0,3,4,1,1,0,0,2
1,5205,11598710423751654092,5205,5205,2018-04-04 14:45:00+00:00,asset__001,run__001,NORMAL,0,1,0.572935,-0.072935,-1.0,1,1,1,-0.572935,1.0,3,4,1,1,0,0,2
2,8195,1628843015739861844,8195,8195,2018-04-06 16:35:00+00:00,asset__001,run__001,NORMAL,0,1,0.567308,-0.067308,-1.0,1,1,1,-0.567308,1.0,2,4,1,1,0,0,2
3,8239,14617586387100313698,8239,8239,2018-04-06 17:19:00+00:00,asset__001,run__001,NORMAL,0,1,0.571852,-0.071852,-1.0,1,1,1,-0.571852,1.0,2,6,1,1,0,0,2
4,8241,12991760080095676024,8241,8241,2018-04-06 17:21:00+00:00,asset__001,run__001,NORMAL,0,1,0.566312,-0.066312,-1.0,1,1,1,-0.566312,1.0,1,6,0,1,1,0,2
5,8248,15244127189313681115,8248,8248,2018-04-06 17:28:00+00:00,asset__001,run__001,NORMAL,0,1,0.567661,-0.067661,-1.0,1,1,1,-0.567661,1.0,1,6,0,1,1,0,2
6,8249,3622191867065128171,8249,8249,2018-04-06 17:29:00+00:00,asset__001,run__001,NORMAL,0,1,0.581498,-0.081498,-1.0,1,1,1,-0.581498,1.0,2,5,1,1,1,0,3
7,10614,6522429923118725157,10614,10614,2018-04-08 08:54:00+00:00,asset__001,run__001,NORMAL,0,1,0.566434,-0.066434,-1.0,1,1,1,-0.566434,1.0,1,4,0,1,0,1,2
8,10615,5800922206161343483,10615,10615,2018-04-08 08:55:00+00:00,asset__001,run__001,NORMAL,0,1,0.572049,-0.072049,-1.0,1,1,1,-0.572049,1.0,1,4,0,1,1,0,2
9,10632,15188046409900994420,10632,10632,2018-04-08 09:12:00+00:00,asset__001,run__001,NORMAL,0,1,0.571604,-0.071604,-1.0,1,1,1,-0.571604,1.0,2,4,1,1,0,0,2


----

In [53]:
row_tracking_03a_stage1_detected_rows_path = save_data(
    stage1_detected_rows_dataframe,
    CASCADE_ROW_TRACKING_DIR,
    f"{DATASET_NAME}__gold__cascade_03a__row_tracking__stage1_detection.csv",
)

row_tracking_03a_stage2_detected_rows_path = save_data(
    stage2_detected_rows_dataframe,
    CASCADE_ROW_TRACKING_DIR,
    f"{DATASET_NAME}__gold__cascade_03a__row_tracking__stage2_detection.csv",
)

row_tracking_03a_stage3_evidence_detection_path = save_data(
    stage3_evidence_rows_dataframe,
    CASCADE_ROW_TRACKING_DIR,
    f"{DATASET_NAME}__gold__cascade_03a__row_tracking__stage3_evidence_detection.csv",
)

row_tracking_03a_final_detected_rows_path = save_data(
    final_detected_rows_dataframe,
    CASCADE_ROW_TRACKING_DIR,
    f"{DATASET_NAME}__gold__cascade_03a__row_tracking__final_detection.csv",
)

wandb.save(str(row_tracking_03a_stage1_detected_rows_path))
wandb.save(str(row_tracking_03a_stage2_detected_rows_path))
wandb.save(str(row_tracking_03a_stage3_evidence_detection_path))
wandb.save(str(row_tracking_03a_final_detected_rows_path))

ledger.add(
    kind="step",
    step="finalize_row_tracking_and_save_artifacts",
    message="Saved cascade 03A row-tracking artifacts.",
    data={
        "row_tracking_cascade_03a_stage1": str(row_tracking_03a_stage1_detected_rows_path),
        "row_tracking_cascade_03a_stage2": str(row_tracking_03a_stage2_detected_rows_path),
        "row_tracking_cascade_03a_stage3_evidence": str(row_tracking_03a_stage3_evidence_detection_path),
        "row_tracking_cascade_03a_final": str(row_tracking_03a_final_detected_rows_path),
    },
    logger=logger,
)

2026-06-16 18:15:04,970 | INFO | capstone.file_io | Saving DataFrame to CSV: /workspace/artifacts/gold/pump/cascade_defaults/row_tracking/pump__gold__cascade_03a__row_tracking__stage1_detection.csv


2026-06-16 18:15:11,171 | INFO | capstone.file_io | Saved: pump__gold__cascade_03a__row_tracking__stage1_detection.csv | shape=(103054, 16) | columns=['meta__row_id', 'meta__record_id', 'time_index', 'event_step', 'event_time', 'meta__asset_id', 'meta__run_id', 'machine_status', 'anomaly_flag', 'stage1_flag', 'stage1_score', 'stage1_decision', 'stage1_pred', 'stage2_raw_flag', 'stage2_flag', 'cascade_final_flag']
2026-06-16 18:15:11,186 | INFO | capstone.file_io | Saving DataFrame to CSV: /workspace/artifacts/gold/pump/cascade_defaults/row_tracking/pump__gold__cascade_03a__row_tracking__stage2_detection.csv
2026-06-16 18:15:15,943 | INFO | capstone.file_io | Saved: pump__gold__cascade_03a__row_tracking__stage2_detection.csv | shape=(70376, 18) | columns=['meta__row_id', 'meta__record_id', 'time_index', 'event_step', 'event_time', 'meta__asset_id', 'meta__run_id', 'machine_status', 'anomaly_flag', 'stage2_flag', 'stage2_score', 'stage2_model_decision', 'stage2_model_pred', 'stage1_flag'

{'ts_utc': '2026-06-16T18:15:26.157799+00:00',
 'stage': 'gold_cascade',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'finalize_row_tracking_and_save_artifacts',
 'message': 'Saved cascade 03A row-tracking artifacts.',
 'why': None,
 'consequence': None,
 'data': {'row_tracking_cascade_03a_stage1': '/workspace/artifacts/gold/pump/cascade_defaults/row_tracking/pump__gold__cascade_03a__row_tracking__stage1_detection.csv',
  'row_tracking_cascade_03a_stage2': '/workspace/artifacts/gold/pump/cascade_defaults/row_tracking/pump__gold__cascade_03a__row_tracking__stage2_detection.csv',
  'row_tracking_cascade_03a_stage3_evidence': '/workspace/artifacts/gold/pump/cascade_defaults/row_tracking/pump__gold__cascade_03a__row_tracking__stage3_evidence_detection.csv',
  'row_tracking_cascade_03a_final': '/workspace/artifacts/gold/pump/cascade_defaults/row_tracking/pump__gold__cascade_03a__row_tracking__final_detection.csv'}}

## Finalize the Ledger and Close the Tracking Run

This step writes the cascade ledger to disk and cleanly closes the experiment tracking run.

By the time I get here, the important modeling and artifact creation work is already complete. So this section is mainly about wrapping up the notebook in a structured way.

In [54]:
ledger.add(
    kind="step",
    step="finalize_cascade_modeling",
    message="Gold cascade modeling notebook complete.",
    data={
        "cascade_variant": CASCADE_VARIANT,
        "stage2_selection_mode": STAGE2_SELECTION_MODE,
        "cascade_defaults_metrics": cascade_metrics,
        "cascade_defaults_results_path_csv": str(CASCADE_RESULTS_PATH_CSV),
        "cascade_defaults_results_path_pickle": str(CASCADE_RESULTS_PATH_PICKLE),
        "cascade_defaults_stage1_model_artifact_path": str(STAGE1_MODEL_ARTIFACT_PATH),
        "cascade_defaults_stage1_models_path": str(STAGE1_MODELS_PATH),
        "cascade_defaults_stage2_model_artifact_path": str(STAGE2_MODEL_ARTIFACT_PATH),
        "cascade_defaults_stage2_models_path": str(STAGE2_MODELS_PATH),
    },
    logger=logger,
)

cascade_ledger_path = (
    GOLD_CASCADE_ARTIFACT_DIRS["lineage"]
    / GOLD_CASCADE_LEDGER_FILE_NAME
)

cascade_ledger_path.parent.mkdir(parents=True, exist_ok=True)
ledger.write_json(cascade_ledger_path)
wandb.save(str(cascade_ledger_path))

wandb_run.finish()

2026-06-16 18:15:27,008 | INFO | capstone.gold.cascade.defaults | LEDGER | {'ts_utc': '2026-06-16T18:15:27.008016+00:00', 'stage': 'gold_cascade', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'finalize_cascade_modeling', 'message': 'Gold cascade modeling notebook complete.', 'why': None, 'consequence': None, 'data': {'cascade_variant': 'default', 'stage2_selection_mode': 'fixed', 'cascade_defaults_metrics': {'model': '3-Stage Cascade', 'stage1_alert_count_all_rows': 103054, 'stage2_alert_count_all_rows': 70376, 'final_alert_count_all_rows': 69362, 'stage1_alert_count_test_rows': 43236, 'stage2_alert_count_test_rows': 25738, 'final_alert_count_test_rows': 24895, 'precision': 0.0030929905603534844, 'recall': 0.652542372881356, 'f1': 0.006156798464798305}, 'cascade_defaults_results_path_csv': '/workspace/artifacts/gold/pump/cascade_defaults/scores/pump__gold__cascade_defaults_results.csv', 'cascade_defaults_results_path_pickle': '/workspace/artifacts/gold/pump/cascade_

----

## Run Final Lineage and Consistency Checks

Before I treat the cascade notebook as complete, I run a final sanity check on the saved cascade results and truth artifacts.

This check verifies things like:
- required lineage columns exist in the results dataframe
- the dataframe truth hash matches the saved cascade truth
- the parent truth hash matches the Gold parent truth
- the saved truth file exists
- the saved metadata points back to the correct cascade truth hash

I like ending with this because it confirms that the cascade output is not only saved, but also internally consistent and traceable.

In [55]:
required_cascade_meta_columns = [
    "meta__truth_hash",
    "meta__parent_truth_hash",
    "meta__pipeline_mode",
]

missing_cascade_meta_columns = [
    column_name
    for column_name in required_cascade_meta_columns
    if column_name not in cascade_results.columns
]
if missing_cascade_meta_columns:
    raise ValueError(
        f"cascade_results is missing required lineage columns: {missing_cascade_meta_columns}"
    )

cascade_results_truth_hash_check = extract_truth_hash(cascade_results)
if cascade_results_truth_hash_check is None:
    raise ValueError("cascade_results does not contain a readable meta__truth_hash value.")

if cascade_results_truth_hash_check != CASCADE_TRUTH_HASH:
    raise ValueError(
        "cascade_results truth hash does not match CASCADE_TRUTH_HASH:\n"
        f"dataframe={cascade_results_truth_hash_check}\n"
        f"record={CASCADE_TRUTH_HASH}"
    )

cascade_parent_values = cascade_results["meta__parent_truth_hash"].dropna().astype(str).unique().tolist()
if not cascade_parent_values:
    raise ValueError("cascade_results is missing populated meta__parent_truth_hash values.")

if len(cascade_parent_values) != 1:
    raise ValueError(f"cascade_results has multiple parent truth hashes: {cascade_parent_values}")

if cascade_parent_values[0] != GOLD_PARENT_TRUTH_HASH:
    raise ValueError(
        "cascade_results parent truth hash does not match GOLD_PARENT_TRUTH_HASH:\n"
        f"dataframe_parent={cascade_parent_values[0]}\n"
        f"gold_parent={GOLD_PARENT_TRUTH_HASH}"
    )

if not Path(cascade_truth_path).exists():
    raise FileNotFoundError(f"Cascade truth file was not created: {cascade_truth_path}")

loaded_cascade_truth: dict[str, Any] = require_mapping(
    load_json(cascade_truth_path),
    "loaded_cascade_truth",
)

loaded_cascade_truth_hash = loaded_cascade_truth.get("truth_hash")

if loaded_cascade_truth_hash != CASCADE_TRUTH_HASH:
    raise ValueError(
        "Saved Cascade truth file hash does not match CASCADE_TRUTH_HASH:\n"
        f"file={loaded_cascade_truth_hash}\n"
        f"expected={CASCADE_TRUTH_HASH}"
    )

loaded_cascade_parent_truth_hash = loaded_cascade_truth.get("parent_truth_hash")

if loaded_cascade_parent_truth_hash != GOLD_PARENT_TRUTH_HASH:
    raise ValueError(
        "Saved Cascade truth file parent hash does not match GOLD_PARENT_TRUTH_HASH:\n"
        f"truth={loaded_cascade_parent_truth_hash}\n"
        f"gold_parent={GOLD_PARENT_TRUTH_HASH}"
    )

saved_cascade_metadata: dict[str, Any] = require_mapping(
    load_json(CASCADE_METADATA_PATH),
    "saved_cascade_metadata",
)

saved_cascade_metadata_truth_hash = saved_cascade_metadata.get("cascade_truth_hash")

if saved_cascade_metadata_truth_hash != CASCADE_TRUTH_HASH:
    raise ValueError(
        "cascade_metadata cascade_truth_hash does not match CASCADE_TRUTH_HASH:\n"
        f"metadata={saved_cascade_metadata_truth_hash}\n"
        f"expected={CASCADE_TRUTH_HASH}"
    )

print("Gold Cascade lineage sanity check passed.")

2026-06-16 18:18:16,098 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/truths/gold_cascade/pump__gold_cascade__truth__70cc8fe26c311e8ff807cdd4c5151cafcf2637f37d197d5cfdf32e400ef6dd97.json
2026-06-16 18:18:16,116 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/gold/pump/cascade_defaults/metadata/pump__gold__cascade_defaults_metadata.json


Gold Cascade lineage sanity check passed.


### Ask

What does this final sanity check really confirm?

### Answer

It confirms that the cascade results can be trusted as a pipeline artifact, not just as notebook output.

A modeling notebook can run successfully and still leave behind broken lineage or mismatched truth metadata. This final check helps guard against that by making sure the dataframe, truth record, and saved metadata all agree with each other.

So this is really a trust check more than a completion check.

In [56]:
#STAGE2_SELECTION_MODE = STAGE2_CFG["fixed"]
STAGE2_FIXED_THRESHOLD_PERCENTILE = float(
    STAGE2_CFG["fixed"]["threshold_percentile"]
)

#print("Stage 2 selection mode:", STAGE2_SELECTION_MODE)
print("Stage 2 fixed threshold percentile:", STAGE2_FIXED_THRESHOLD_PERCENTILE)

Stage 2 fixed threshold percentile: 99.0


#### Ask

This cell performs the next transformation in `Ask`. Review the output or logged message before relying on this result downstream.

In [57]:
cascade_results

,meta__asset_id,meta__dataset,meta__episode_id,meta__event_id,meta__ingested_at_utc,meta__parent_truth_hash,meta__pipeline_mode,meta__record_id,meta__run_id,meta__source_file,meta__source_row_id,meta__split,meta__truth_hash,event_time,event_step,time_index,event_date,anomaly_flag,is_anomaly,is_normal,status_normal_value,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,...,sensor_47__delta_abnormal_flag,sensor_47__any_abnormal_flag,sensor_48__value_deviation,sensor_48__delta_deviation,sensor_48__value_abnormal_flag,sensor_48__delta_abnormal_flag,sensor_48__any_abnormal_flag,sensor_49__value_deviation,sensor_49__delta_deviation,sensor_49__value_abnormal_flag,sensor_49__delta_abnormal_flag,sensor_49__any_abnormal_flag,sensor_51__value_deviation,sensor_51__delta_deviation,sensor_51__value_abnormal_flag,sensor_51__delta_abnormal_flag,sensor_51__any_abnormal_flag,normal_value_abnormal_sensor_count,normal_delta_abnormal_sensor_count,normal_total_abnormal_sensor_count,normal_training_quality_class,is_clean_normal_for_training,final_row_quality_class,row_is_clean_normal,row_is_suspect_normal,row_is_exclude_from_normal_training,machine_status__profiled,meta__row_id,meta__is_train_flag,stage1_score,stage1_decision,stage1_pred,stage1_flag,stage1_threshold,stage1_threshold_percentile,stage2_model_score,stage2_model_decision,stage2_model_pred,stage2_model_flag,stage2_score,stage2_raw_flag,stage2_flag,stage3_profile_breach_count,stage3_secondary_breach_count,stage3_persistence_flag,stage3_drift_flag,stage3_profile_breach_flag,stage3_corroboration_flag,stage3_rule_evidence_count,cascade_final_flag
0,asset__001,pump,0,pump:asset__001:run__001:0,2026-06-14 20:37:14.939771+00:00,1b39755619d21614721b74a063b90aceddebaa1f4b8573...,batch,14598431322315673869,run__001,sensor.csv,0,train,70cc8fe26c311e8ff807cdd4c5151cafcf2637f37d197d...,2018-04-01 00:00:00+00:00,0,0,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,0.140011,-0.695652,0.464286,1.269226,0.108436,-0.168590,-0.369557,0.124978,0.736837,-0.266728,-1.260975,0.925032,-0.034035,-0.123905,-0.461029,-0.672626,0.304680,0.142472,-0.295277,-0.461054,-0.091023,-0.172259,-0.121836,0.099551,-0.311720,-0.479814,-0.666538,-1.737734,-0.546495,...,False,False,0.716570,NaN,False,False,False,2.780487,NaN,False,False,False,0.275862,NaN,False,False,False,4,0,4,clean,True,clean,True,False,False,normal_clean,0,True,0.409368,0.090632,1,0,0.488648,95.0,NaN,NaN,NaN,NaN,NaN,0,0,1,2,0,0,0,1,1,0
1,asset__001,pump,0,pump:asset__001:run__001:1,2026-06-14 20:37:14.939771+00:00,1b39755619d21614721b74a063b90aceddebaa1f4b8573...,batch,15954729095895098000,run__001,sensor.csv,1,train,70cc8fe26c311e8ff807cdd4c5151cafcf2637f37d197d...,2018-04-01 00:01:00+00:00,1,1,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,0.140011,-0.695652,0.464286,1.269226,0.108436,-0.168590,-0.369557,0.124978,0.736837,-0.266728,-1.260975,0.925032,-0.034035,-0.123905,-0.461029,-0.672626,0.304680,0.142472,-0.295277,-0.461054,-0.091023,-0.172259,-0.121836,0.099551,-0.311720,-0.479814,-0.666538,-1.737734,-0.546495,...,False,False,0.716570,0.000000,False,False,False,2.780487,0.000000,False,False,False,0.275862,0.000000,False,False,False,4,0,4,clean,True,clean,True,False,False,normal_clean,1,True,0.409368,0.090632,1,0,0.488648,95.0,NaN,NaN,NaN,NaN,NaN,0,0,1,2,0,0,0,1,1,0
2,asset__001,pump,0,pump:asset__001:run__001:2,2026-06-14 20:37:14.939771+00:00,1b39755619d21614721b74a063b90aceddebaa1f4b8573...,batch,10041703297090838359,run__001,sensor.csv,2,train,70cc8fe26c311e8ff807cdd4c5151cafcf2637f37d197d...,2018-04-01 00:02:00+00:00,2,2,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,-0.280002,-0.565216,0.464286,1.307690,0.578311,-0.537436,-0.630443,-0.078128,0.859651,-0.666728,-1.160344,1.006862,0.146473,-0.108279,0.230324,-0.259305,-0.439206,-0.506023,

#### Review intermediate output

This cell displays an intermediate checkpoint. Use the output to confirm that the expected rows, columns, dtypes, or summary values are present before continuing.

In [58]:
display(
    cascade_results.loc[
        cascade_results["time_index"].between(17130, 17170),
        [
            "time_index",
            "machine_status",
            "stage1_flag",
            "stage2_score",
            "stage2_raw_flag",
            "stage2_flag",
            "cascade_final_flag",
        ]
    ]
)

,time_index,machine_status,stage1_flag,stage2_score,stage2_raw_flag,stage2_flag,cascade_final_flag
17130,17130,NORMAL,1,0.558019,0,0,0
17131,17131,NORMAL,1,0.556620,0,0,0
17132,17132,NORMAL,1,0.566368,1,1,0
17133,17133,NORMAL,1,0.555894,0,0,0
17134,17134,NORMAL,1,0.551118,0,0,0
17135,17135,NORMAL,1,0.546106,0,0,0
17136,17136,NORMAL,1,0.550059,0,0,0
17137,17137,NORMAL,1,0.555598,0,0,0
17138,17138,NORMAL,1,0.558415,0,0,0
17139,17139,NORMAL,1,0.548274,0,0,0


#### Define first-alert lookup helper

This cell defines helper logic used by the surrounding `Ask` section. Keeping the helper directly before its use makes the notebook assumptions visible and easier to audit.

In [59]:
def get_first_alert_index(df, col):
    if col not in df.columns:
        return None
    rows = df.loc[df[col].fillna(0).astype(int) == 1, "time_index"]
    return int(rows.iloc[0]) if not rows.empty else None



#### Define first-alert lookup helper

This cell performs the next transformation in `Ask`. Review the output or logged message before relying on this result downstream.

In [60]:
stage_alerts = {
    "stage1_flag": get_first_alert_index(cascade_results, "stage1_flag"),
    "stage2_raw_flag": get_first_alert_index(cascade_results, "stage2_raw_flag"),
    "stage2_flag": get_first_alert_index(cascade_results, "stage2_flag"),
    "cascade_final_flag": get_first_alert_index(cascade_results, "cascade_final_flag"),
}
stage_alerts

{'stage1_flag': 260,
 'stage2_raw_flag': 5202,
 'stage2_flag': 5202,
 'cascade_final_flag': 5202}

#### Ask

This cell performs the next transformation in `Ask`. Review the output or logged message before relying on this result downstream.

In [61]:
{
    "stage2_score_non_null_count": int(cascade_results["stage2_score"].notna().sum()),
    "row_count": int(len(cascade_results)),
}

{'stage2_score_non_null_count': 103054, 'row_count': 220320}

#### Review intermediate output

This cell displays an intermediate checkpoint. Use the output to confirm that the expected rows, columns, dtypes, or summary values are present before continuing.

In [62]:
display(
    cascade_results.loc[
        cascade_results["time_index"].between(10380, 10460),
        [
            "time_index",
            "machine_status",
            "stage1_flag",
            "stage2_score",
            "stage2_raw_flag",
            "stage2_flag",
            "cascade_final_flag",
        ]
    ]
)

,time_index,machine_status,stage1_flag,stage2_score,stage2_raw_flag,stage2_flag,cascade_final_flag
10380,10380,NORMAL,0,NaN,0,0,0
10381,10381,NORMAL,0,NaN,0,0,0
10382,10382,NORMAL,0,NaN,0,0,0
10383,10383,NORMAL,0,NaN,0,0,0
10384,10384,NORMAL,0,NaN,0,0,0
...,...,...,...,...,...,...,...
10456,10456,NORMAL,1,0.541393,0,0,0
10457,10457,NORMAL,1,0.540058,0,0,0
10458,10458,NORMAL,1,0.538699,0,0,0
10459,10459,NORMAL,1,0.551897,0,0,0


----


# Gold Cascade SQL Write Cell
Target:
- gold.anomaly_detection_scores

Purpose:
- Persist final cascade anomaly scores, flags, and rule/evidence fields.


In [63]:
WRITE_TO_POSTGRES = True

CASCADE_SQL_MODEL_STAGE = "cascade_default_final"

if WRITE_TO_POSTGRES:
    gold_cascade_sql_summary_dataframe = write_gold_cascade_scores_sql(
        engine=engine,
        capstone_schema=CAPSTONE_SCHEMA,
        dataset_id=DATASET_ID,
        run_id=RUN_ID,
        notebook_globals=globals(),
        dataframe=cascade_results,
        dataset_name=globals().get("DATASET_NAME", DATASET_ID),
        model_stage=CASCADE_SQL_MODEL_STAGE,
    )

    display(gold_cascade_sql_summary_dataframe)

else:
    print("Postgres write skipped.")

Using supplied dataframe -> 220,320 rows x 362 columns
Deleted 220,320 existing rows from gold.anomaly_detection_scores for cascade_isolation_forest_rule_confirmation/cascade_default_final.
Wrote 220,320 rows.
Upserted pipeline run metadata: run__001__gold_cascade_default_final_modeling
Logged DQ event: gold.anomaly_detection_scores | cascade_default_final_sql_write | pass


,model_name,model_stage,row_count,alert_count
0,cascade_isolation_forest_rule_confirmation,cascade_default_final,220320,69362


## Summary

This notebook preserves the current analytical behavior while documenting the role of the Gold step in the capstone workflow. The code cells above should continue to produce the same configured outputs, artifacts, logger messages, and ledger entries as before this documentation pass.


## Next Stage

Gold 03b continues cascade tuning using the same Gold modeling foundation.
